In [14]:
from scripts.dataset import context_a, context_b, context_c
import pandas as pd
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)

In [9]:
domain_df = pd.concat([
    context_a.assign(domain="A"),
    context_b.assign(domain="B"),
    context_c.assign(domain="C"),
], ignore_index=True)

In [10]:
structural_features = [
    "message_index",
    "previous_messages",
    "previous_tool_calls",
    "previous_tool_results",
    "previous_user_messages",
    "previous_assistant_messages",
    "context_char_length",
    "context_word_count",
    "current_char_length",
    "current_word_count",
    "is_tool_call",
]

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score

X_domain_struct = domain_df[structural_features]
y_domain = domain_df["domain"]

domain_structural_model = Pipeline([
    (
        "scaler",
        StandardScaler(),
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=3000,
            random_state=42,
        ),
    ),
])

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

scores = cross_val_score(
    domain_structural_model,
    X_domain_struct,
    y_domain,
    cv=cv,
    scoring="accuracy",
)

print(
    f"Structural-only dataset prediction: "
    f"{scores.mean():.3f} ± {scores.std():.3f}"
)

Structural-only dataset prediction: 0.969 ± 0.005


That is a very important result:

Structural-only dataset prediction: 0.969 ± 0.005

It means that even after completely removing the text, a simple model can determine whether a sample belongs to A, B, or C with about 96.9% accuracy using only trajectory structure.

What this tells you

Previously you found:

Text-based dataset prediction:       1.000 ± 0.000
Structural-only dataset prediction:  0.969 ± 0.005
Random baseline:                     ~0.333

So the domain shift is not mainly caused by vocabulary or tool names.

The datasets differ strongly at a structural level.

This agrees with the statistics you already observed:

Property	A	B	C
Avg context words	488	92	28
Median context words	345	52	14
Avg current words	53	54	25
Tool-call %	62%	53%	67%
Error %	24%	31%	22%

A classifier can essentially say:

Very long context
+ particular trajectory position/counts
        ↓
probably dataset A


Medium context
+ different trajectory structure
        ↓
probably dataset B


Very short context
+ high tool-call frequency
        ↓
probably dataset C

It doesn't even need to know what the messages say.

This changes the normalization experiment slightly

I would still run text normalization, because it's useful to quantify lexical shortcuts.

But don't expect:

source accuracy:
1.00 → 0.40

because structure alone already gives you 0.969.

Even perfect text normalization cannot eliminate structural dataset identity.

Your expected experiment now looks more like:

Raw text               1.000
Normalized text            ?
Structure only         0.969

If normalized text remains 0.98–1.00, that's not surprising.

In [ ]:
# More important next experiment: identify WHICH structural features reveal the dataset

from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

structural_features = [
    "message_index",
    "previous_messages",
    "previous_tool_calls",
    "previous_tool_results",
    "previous_user_messages",
    "previous_assistant_messages",
    "context_char_length",
    "context_word_count",
    "current_char_length",
    "current_word_count",
    "is_tool_call",
]

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

results = []

for feature in structural_features:

    model = Pipeline([
        (
            "scaler",
            StandardScaler(),
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=3000,
                random_state=42,
            ),
        ),
    ])

    scores = cross_val_score(
        model,
        domain_df[[feature]],
        domain_df["domain"],
        cv=cv,
        scoring="accuracy",
    )

    results.append({
        "feature": feature,
        "mean_accuracy": scores.mean(),
        "std_accuracy": scores.std(),
    })

structural_feature_results = (
    pd.DataFrame(results)
    .sort_values(
        "mean_accuracy",
        ascending=False,
    )
    .reset_index(drop=True)
)

structural_feature_results

,feature,mean_accuracy,std_accuracy
0,context_word_count,0.649180,0.013375
1,context_char_length,0.634939,0.013429
2,previous_assistant_messages,0.630433,0.009116
3,current_word_count,0.528123,0.011961
4,previous_tool_calls,0.516931,0.000289
5,previous_tool_results,0.516931,0.000289
6,previous_user_messages,0.516931,0.000289
7,is_tool_call,0.516931,0.000289
8,current_char_length,0.509229,0.026807
9,previous_messages,0.486413,0.005643


In [11]:
results = []

for removed_feature in structural_features:

    remaining = [
        f
        for f in structural_features
        if f != removed_feature
    ]

    model = Pipeline([
        (
            "scaler",
            StandardScaler(),
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=3000,
                random_state=42,
            ),
        ),
    ])

    scores = cross_val_score(
        model,
        domain_df[remaining],
        domain_df["domain"],
        cv=cv,
        scoring="accuracy",
    )

    results.append({
        "removed_feature": removed_feature,
        "mean_accuracy": scores.mean(),
        "std_accuracy": scores.std(),
    })

ablation_results = (
    pd.DataFrame(results)
    .sort_values("mean_accuracy")
    .reset_index(drop=True)
)

ablation_results

,removed_feature,mean_accuracy,std_accuracy
0,message_index,0.756578,0.013085
1,previous_messages,0.956547,0.003769
2,previous_assistant_messages,0.957273,0.004276
3,is_tool_call,0.963377,0.004375
4,context_word_count,0.963813,0.003661
5,previous_user_messages,0.968754,0.004620
6,previous_tool_calls,0.969190,0.004253
7,previous_tool_results,0.969190,0.004253
8,current_word_count,0.969191,0.003887
9,context_char_length,0.969626,0.004371


In [12]:
from sklearn.model_selection import GroupKFold

group_cv = GroupKFold(
    n_splits=5
)

scores = cross_val_score(
    domain_structural_model,
    domain_df[structural_features],
    domain_df["domain"],
    groups=domain_df["group_id"],
    cv=group_cv,
    scoring="accuracy",
)

print(
    "Grouped structural source prediction: "
    f"{scores.mean():.3f} ± {scores.std():.3f}"
)

Grouped structural source prediction: 0.971 ± 0.004


Yes — this makes the conclusion considerably stronger. The grouped result is especially important:

> **Grouped structural dataset-source accuracy = 0.971 ± 0.004**

Because trajectories are separated between folds, this is not explained by seeing other messages from the same trajectory. The structural differences are genuinely characteristic of the datasets.

### What the experiments tell you

Your evidence now looks like this:

| Experiment                        |   Source accuracy |
| --------------------------------- | ----------------: |
| Random baseline                   |            ~0.333 |
| Text TF-IDF                       |         **1.000** |
| All structural features           |         **0.969** |
| Grouped structural features       | **0.971 ± 0.004** |
| Context word count alone          |             0.649 |
| Context char length alone         |             0.635 |
| Previous assistant messages alone |             0.630 |

So both **language/domain** and **trajectory structure** reveal dataset identity extremely strongly.

The grouped result being essentially identical to the original `0.969` is particularly reassuring methodologically.

---

## There is a very interesting result in your ablation

At first glance, the single-feature table says:

```text
context_word_count          0.649
context_char_length         0.635
previous_assistant_messages 0.630
message_index               0.481
```

So you might conclude that `context_word_count` is the most important feature.

But your leave-one-out experiment reveals something more interesting:

| Removed feature             | Accuracy after removal |
| --------------------------- | ---------------------: |
| **message_index**           |              **0.757** |
| previous_messages           |                  0.957 |
| previous_assistant_messages |                  0.957 |
| is_tool_call                |                  0.963 |
| context_word_count          |                  0.964 |
| everything                  |              **0.969** |

Removing `message_index` causes:

```text
0.969 → 0.757
```

That's a huge drop of about **21 percentage points**.

Yet `message_index` alone only gives:

```text
0.481
```

This is not contradictory.

It means `message_index` has a strong **interaction with other structural variables**.

For example, the model may learn relationships like:

```text
message_index = 10
+
context_words = 400
        ↓
likely A

message_index = 10
+
context_words = 30
        ↓
likely C
```

`message_index` isn't necessarily informative enough by itself, but combined with trajectory length/context statistics it becomes extremely useful.

That's a good finding.

---

# What this means for your reliability model

This raises an important concern about your existing structural features.

Your reliability model contains features such as:

```python
message_index
previous_messages
previous_tool_calls
previous_tool_results
previous_user_messages
previous_assistant_messages
context_char_length
context_word_count
current_char_length
current_word_count
is_tool_call
```

You have now demonstrated that those same features encode dataset identity with:

> **97.1% grouped accuracy.**

Therefore when your reliability model uses them, it has access to extremely strong domain information.

That doesn't mean you should simply delete all structural features. Some of them may genuinely be useful for detecting errors.

Instead, you need to determine:

> **Which structural features improve reliability prediction because they represent general agent behavior, and which primarily act as domain identifiers?**

---

# Your next experiment should now be reliability ablation

This is more important than further source-classification experiments.

Take your **LinearSVC**, because it is currently your strongest model, and create several variants.

I recommend these five:

| Experiment | Text | Structural                         |
| ---------- | ---- | ---------------------------------- |
| A          | Raw  | All                                |
| B          | Raw  | None                               |
| C          | Raw  | Remove `message_index`             |
| D          | Raw  | Remove length features             |
| E          | Raw  | Minimal domain-resistant structure |

For the minimal structural version, try:

```python
minimal_structural = [
    "is_tool_call",
]
```

And perhaps another:

```python
behavioral_structural = [
    "previous_tool_calls",
    "previous_tool_results",
    "previous_user_messages",
    "previous_assistant_messages",
    "is_tool_call",
]
```

Then evaluate **both**:

```text
IID Macro F1
AND
A+B → C
A+C → B
B+C → A
```

You're looking for a tradeoff such as:

```text
                        IID       OOD
All structural          .655      .367
No structural           .630      .400
Minimal structural      .640      .430
```

Those are just illustrative numbers, but if you observe that pattern it would be a major result.

You'd have evidence that:

> Some features improve conventional held-out performance while reducing domain generalization because they encode benchmark-specific structural patterns.

That's exactly the kind of shortcut-learning hypothesis your current analysis suggests.

---

## I would also still do normalization

But its purpose is now clearer.

You have identified **two separate shortcut channels**:

```text
            Dataset identity
                  │
          ┌───────┴───────┐
          ↓               ↓
     lexical shift    structural shift
          │               │
    source=100%       source=97.1%
```

So text normalization alone cannot solve the problem.

You eventually want to attack both.

For text:

```text
IDs        → <ID>
numbers    → <NUM>
dates      → <DATE>
URLs       → <URL>
...
```

For structure, use the ablation experiments above to determine which features should be retained.

---

# One caution about the source-classification metric

Don't interpret `97.1%` as:

> "97.1% of structural information is dataset-specific."

It doesn't mean that.

It means:

> Given these structural features, a classifier can correctly identify the source dataset for approximately 97.1% of examples.

Similarly, it doesn't prove that your reliability classifier **is actually using all of those shortcuts**.

That's why the next reliability ablation is necessary.

You've established **shortcut availability**.

Now you need to establish **shortcut dependence**:

```text
Step 1 — Can structural features identify domain?
YES → 97.1%

              ↓

Step 2 — Does removing domain-identifying structure
         improve OOD reliability classification?
         ← NEXT EXPERIMENT

              ↓

Step 3 — Normalize lexical domain signals

              ↓

Step 4 — Combine best normalization +
         best structural subset

              ↓

Step 5 — Only then consider a new model
```

I would do **Step 2 next**, before training another BERT. It directly connects your dataset diagnosis to your actual reliability-classification objective.


In [15]:
all_structural = [
    "message_index",
    "previous_messages",
    "previous_tool_calls",
    "previous_tool_results",
    "previous_user_messages",
    "previous_assistant_messages",
    "context_char_length",
    "context_word_count",
    "current_char_length",
    "current_word_count",
    "is_tool_call",
]

no_message_index = [
    feature
    for feature in all_structural
    if feature != "message_index"
]

no_length_features = [
    feature
    for feature in all_structural
    if feature not in {
        "context_char_length",
        "context_word_count",
        "current_char_length",
        "current_word_count",
    }
]

behavioral_structural = [
    "previous_tool_calls",
    "previous_tool_results",
    "previous_user_messages",
    "previous_assistant_messages",
    "is_tool_call",
]

minimal_structural = [
    "is_tool_call",
]

In [16]:
feature_experiments = {
    "all_structural": all_structural,

    "no_structural": [],

    "no_message_index": no_message_index,

    "no_length_features": no_length_features,

    "behavioral_structural": behavioral_structural,

    "minimal_structural": minimal_structural,
}

In [17]:
def make_reliability_model(
    structural_features,
    categorical_features=None,
):
    if categorical_features is None:
        categorical_features = [
            "current_role",
        ]

    transformers = []

    # Structural numeric features
    if structural_features:
        transformers.append(
            (
                "numeric",
                StandardScaler(),
                structural_features,
            )
        )

    # Keep role unless you explicitly want to ablate it later
    transformers.append(
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features,
        )
    )

    # Current text
    transformers.append(
        (
            "current_text",
            TfidfVectorizer(
                max_features=5000,
                ngram_range=(1, 2),
                min_df=2,
            ),
            "current_text",
        )
    )

    # Previous context
    transformers.append(
        (
            "context_text",
            TfidfVectorizer(
                max_features=5000,
                ngram_range=(1, 2),
                min_df=2,
            ),
            "context_text",
        )
    )

    preprocessor = ColumnTransformer(
        transformers
    )

    return Pipeline([
        (
            "preprocessor",
            preprocessor,
        ),
        (
            "classifier",
            LinearSVC(
                C=2.0,
                max_iter=10000,
                random_state=42,
            ),
        ),
    ])

In [18]:
def evaluate_iid(
    model,
    X_train,
    y_train,
    X_test,
    y_test,
):
    model.fit(
        X_train,
        y_train,
    )

    pred = model.predict(
        X_test
    )

    return {
        "accuracy":
            accuracy_score(
                y_test,
                pred,
            ),

        "macro_f1":
            f1_score(
                y_test,
                pred,
                average="macro",
            ),

        "error_precision":
            precision_score(
                y_test,
                pred,
                labels=[-1],
                average="macro",
                zero_division=0,
            ),

        "error_recall":
            recall_score(
                y_test,
                pred,
                labels=[-1],
                average="macro",
                zero_division=0,
            ),

        "error_f1":
            f1_score(
                y_test,
                pred,
                labels=[-1],
                average="macro",
                zero_division=0,
            ),
    }

In [20]:
from scripts.dataset import X_train, X_test, y_train, y_test

iid_results = []

for experiment_name, structural_features in feature_experiments.items():

    print(
        f"Running IID: {experiment_name}"
    )

    model = make_reliability_model(
        structural_features
    )

    metrics = evaluate_iid(
        model=model,
        X_train=X_train,
        y_train=y_train,
        X_test=X_test,
        y_test=y_test,
    )

    iid_results.append({
        "experiment":
            experiment_name,

        **metrics,
    })

Running IID: all_structural


/Users/user/proj/agent-reliability-ml/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Running IID: no_structural
Running IID: no_message_index
Running IID: no_length_features
Running IID: behavioral_structural
Running IID: minimal_structural


In [21]:
iid_results_df = (
    pd.DataFrame(iid_results)
    .sort_values(
        "macro_f1",
        ascending=False,
    )
    .reset_index(drop=True)
)

iid_results_df

,experiment,accuracy,macro_f1,error_precision,error_recall,error_f1
0,no_structural,0.840288,0.641047,0.775510,0.756219,0.765743
1,minimal_structural,0.840288,0.641047,0.775510,0.756219,0.765743
2,all_structural,0.844604,0.640319,0.782051,0.758706,0.770202
3,no_message_index,0.844604,0.639143,0.783505,0.756219,0.769620
4,no_length_features,0.841727,0.636922,0.776923,0.753731,0.765152
5,behavioral_structural,0.842446,0.636290,0.780362,0.751244,0.765526


In [22]:
def prepare_reliability_xy(
    df,
    structural_features,
):
    df = df.copy()

    df["current_text"] = (
        df["current_text"]
        .fillna("")
        .astype(str)
    )

    df["context_text"] = (
        df["context_text"]
        .fillna("")
        .astype(str)
    )

    df["current_role"] = (
        df["current_role"]
        .fillna("UNKNOWN")
        .astype(str)
    )

    X = df[
        structural_features
        + [
            "current_role",
            "current_text",
            "context_text",
        ]
    ].copy()

    y = (
        df["label"]
        .astype(int)
        .copy()
    )

    return X, y

In [23]:
def evaluate_cross_dataset(
    train_dfs,
    test_df,
    structural_features,
):
    train_df = pd.concat(
        train_dfs,
        ignore_index=True,
    )

    X_train_cross, y_train_cross = (
        prepare_reliability_xy(
            train_df,
            structural_features,
        )
    )

    X_test_cross, y_test_cross = (
        prepare_reliability_xy(
            test_df,
            structural_features,
        )
    )

    model = make_reliability_model(
        structural_features
    )

    model.fit(
        X_train_cross,
        y_train_cross,
    )

    pred = model.predict(
        X_test_cross
    )

    return {
        "accuracy":
            accuracy_score(
                y_test_cross,
                pred,
            ),

        "macro_f1":
            f1_score(
                y_test_cross,
                pred,
                average="macro",
            ),

        "error_precision":
            precision_score(
                y_test_cross,
                pred,
                labels=[-1],
                average="macro",
                zero_division=0,
            ),

        "error_recall":
            recall_score(
                y_test_cross,
                pred,
                labels=[-1],
                average="macro",
                zero_division=0,
            ),

        "error_f1":
            f1_score(
                y_test_cross,
                pred,
                labels=[-1],
                average="macro",
                zero_division=0,
            ),
    }

In [24]:
from scripts.dataset import context_a, context_b, context_c

cross_results = []

cross_splits = [
    (
        "A+B -> C",
        [context_a, context_b],
        context_c,
    ),

    (
        "A+C -> B",
        [context_a, context_c],
        context_b,
    ),

    (
        "B+C -> A",
        [context_b, context_c],
        context_a,
    ),
]

In [25]:
for experiment_name, structural_features in feature_experiments.items():

    print("\n" + "=" * 70)
    print(experiment_name)
    print("=" * 70)

    for split_name, train_dfs, test_df in cross_splits:

        print(
            f"Running {split_name}"
        )

        metrics = evaluate_cross_dataset(
            train_dfs=train_dfs,
            test_df=test_df,
            structural_features=structural_features,
        )

        cross_results.append({
            "feature_experiment":
                experiment_name,

            "split":
                split_name,

            **metrics,
        })


all_structural
Running A+B -> C
Running A+C -> B
Running B+C -> A

no_structural
Running A+B -> C
Running A+C -> B
Running B+C -> A

no_message_index
Running A+B -> C
Running A+C -> B
Running B+C -> A

no_length_features
Running A+B -> C
Running A+C -> B
Running B+C -> A

behavioral_structural
Running A+B -> C
Running A+C -> B
Running B+C -> A

minimal_structural
Running A+B -> C
Running A+C -> B
Running B+C -> A


In [26]:
cross_results_df = pd.DataFrame(
    cross_results
)

cross_results_df[
    [
        "feature_experiment",
        "split",
        "macro_f1",
        "error_f1",
        "error_recall",
    ]
]

,feature_experiment,split,macro_f1,error_f1,error_recall
0,all_structural,A+B -> C,0.292418,0.338619,0.636842
1,all_structural,A+C -> B,0.378908,0.517923,0.754955
2,all_structural,B+C -> A,0.416429,0.464286,0.581006
3,no_structural,A+B -> C,0.266439,0.357999,0.759649
4,no_structural,A+C -> B,0.302510,0.335431,0.408108
5,no_structural,B+C -> A,0.388097,0.418391,0.508380
6,no_message_index,A+B -> C,0.298005,0.334933,0.612281
7,no_message_index,A+C -> B,0.370783,0.508280,0.718919
8,no_message_index,B+C -> A,0.415024,0.460850,0.575419
9,no_length_features,A+B -> C,0.290980,0.341215,0.645614


In [27]:
cross_summary = (
    cross_results_df
    .groupby(
        "feature_experiment"
    )
    .agg(
        mean_cross_macro_f1=(
            "macro_f1",
            "mean"
        ),

        std_cross_macro_f1=(
            "macro_f1",
            "std"
        ),

        mean_error_f1=(
            "error_f1",
            "mean"
        ),

        mean_error_recall=(
            "error_recall",
            "mean"
        ),
    )
    .sort_values(
        "mean_cross_macro_f1",
        ascending=False,
    )
    .reset_index()
)

cross_summary

,feature_experiment,mean_cross_macro_f1,std_cross_macro_f1,mean_error_f1,mean_error_recall
0,all_structural,0.362585,0.063597,0.440276,0.657601
1,no_message_index,0.361271,0.059087,0.434688,0.635540
2,behavioral_structural,0.361262,0.060278,0.424273,0.588666
3,no_length_features,0.359132,0.063316,0.431641,0.620097
4,no_structural,0.319016,0.062486,0.370607,0.558712
5,minimal_structural,0.318949,0.062513,0.370607,0.558712


In [28]:
final_ablation = (
    iid_results_df[
        [
            "experiment",
            "macro_f1",
        ]
    ]
    .rename(
        columns={
            "experiment":
                "feature_experiment",

            "macro_f1":
                "iid_macro_f1",
        }
    )
    .merge(
        cross_summary,
        on="feature_experiment",
        how="left",
    )
)

final_ablation[
    "iid_to_ood_drop"
] = (
    final_ablation[
        "iid_macro_f1"
    ]
    -
    final_ablation[
        "mean_cross_macro_f1"
    ]
)

final_ablation.sort_values(
    "mean_cross_macro_f1",
    ascending=False,
)

,feature_experiment,iid_macro_f1,mean_cross_macro_f1,std_cross_macro_f1,mean_error_f1,mean_error_recall,iid_to_ood_drop
2,all_structural,0.640319,0.362585,0.063597,0.440276,0.657601,0.277734
3,no_message_index,0.639143,0.361271,0.059087,0.434688,0.635540,0.277873
5,behavioral_structural,0.636290,0.361262,0.060278,0.424273,0.588666,0.275027
4,no_length_features,0.636922,0.359132,0.063316,0.431641,0.620097,0.277790
0,no_structural,0.641047,0.319016,0.062486,0.370607,0.558712,0.322032
1,minimal_structural,0.641047,0.318949,0.062513,0.370607,0.558712,0.322099


These results answer the shortcut question quite well, and there is one slightly surprising conclusion:

> **The structural features strongly identify the dataset, but removing them does not improve cross-dataset reliability. It makes generalization worse.**

So structural features are dataset-specific, but they are **not merely harmful shortcuts**.

### What happened

Your IID results are almost identical across configurations:

| Features         | IID Macro F1 |
| ---------------- | -----------: |
| No structural    |    **0.641** |
| Minimal          |    **0.641** |
| All structural   |        0.640 |
| No message index |        0.639 |
| No length        |        0.637 |
| Behavioral       |        0.636 |

That's interesting by itself. Structural features barely matter for ordinary IID Macro F1.

But cross-domain results tell a different story:

| Features           | Mean OOD Macro F1 | Error Recall |
| ------------------ | ----------------: | -----------: |
| **All structural** |         **0.363** |    **0.658** |
| No message index   |             0.361 |        0.636 |
| Behavioral         |             0.361 |        0.589 |
| No length          |             0.359 |        0.620 |
| No structural      |         **0.319** |        0.559 |
| Minimal            |         **0.319** |        0.559 |

Removing all structural information causes OOD Macro F1 to fall:

```text
0.363 → 0.319
```

That's about a **12% relative decrease**.

More importantly for reliability detection, error recall falls:

```text
0.658 → 0.559
```

So the structural information actually helps detect errors in unseen datasets.

## That changes our hypothesis

Before this experiment, we had:

```text
Structural features
       ↓
predict dataset with 97.1% accuracy
       ↓
Maybe they're harmful dataset shortcuts?
```

Now you've tested that hypothesis directly.

The evidence says:

```text
Structural features
       ↓
contain strong dataset identity
       BUT
       ↓
also contain useful reliability information
       ↓
removing them hurts OOD performance
```

That's an important distinction.

**Dataset-identifying information is not automatically bad information.**

For example, `previous_tool_calls` might vary systematically between A/B/C, but the number of previous tool interactions may still genuinely help determine whether a current action is sensible.

---

### `message_index` is especially interesting

Earlier, removing `message_index` from the **dataset-source classifier** caused source accuracy to collapse:

```text
0.969 → 0.757
```

So it participates strongly in identifying datasets.

But removing it from the reliability classifier changes:

```text
IID:
0.6403 → 0.6391

OOD:
0.3626 → 0.3613
```

Essentially nothing.

That's excellent evidence that:

> A feature can strongly encode dataset identity without necessarily being responsible for the reliability model's OOD failure.

That's exactly why doing the reliability ablation was necessary.

---

## Your best configuration remains all features

For the moment, I would keep:

```python
all_structural
+ current_role
+ current_text
+ context_text
```

You have no experimental reason to remove the structural features.

In fact, your results suggest the opposite.

The strongest cross-domain configuration is:

```text
All structural
Mean OOD Macro F1 = 0.363
Mean error recall  = 0.658
```

And the highest error F1 is also all structural:

```text
0.440
```

So keep them.

---

# Now move to lexical normalization

This becomes the next logical experiment.

We've tested:

> Are structural shortcuts causing the OOD problem?

Answer:

> **Not primarily. Removing them makes things worse.**

Now test:

> Is the model relying too heavily on domain-specific lexical information?

This is particularly plausible because your text source classifier achieved:

```text
A/B/C identification = 100%
```

Do **not** remove all domain semantics. Start with conservative normalization.

I recommend testing three levels:

```text
RAW
 ↓
LIGHT NORMALIZATION
 IDs / dates / URLs / numbers
 ↓
TOOL NORMALIZATION
 + tool names
```

Then measure both:

```text
Dataset-source accuracy
Reliability OOD Macro F1
```

The desired result isn't necessarily the lowest source accuracy. What matters is reliability generalization.

For example:

| Representation   | Source accuracy | IID F1 |  OOD F1 |
| ---------------- | --------------: | -----: | ------: |
| Raw              |           1.000 |   .640 |    .363 |
| Light normalized |             .99 |    .63 | **.40** |
| Tool normalized  |             .85 |    .60 |     .38 |

In that hypothetical case, **light normalization wins**, even though source identification remains very high.

The optimization target is OOD reliability, **not hiding dataset identity at all costs**.

---

## One thing to note about your current LinearSVC score

Earlier your optimized final LinearSVC gave:

```text
Macro F1 ≈ 0.655
```

while this ablation gives:

```text
All structural ≈ 0.640
```

Don't silently replace your previous final result with `0.640`.

The pipelines/configurations are apparently not perfectly identical—possibly categorical features, TF-IDF settings, feature definitions, split preparation, or another preprocessing difference.

For your final report, you should eventually rerun the winning configurations through **one canonical evaluation pipeline** so every model comparison is apples-to-apples.

That's not urgent right now, but do it before finalizing results.

### Where you are now

Your experimental reasoning has become:

```text
OOD collapse discovered
        ↓
A/B/C text highly different
source prediction = 100%
        ↓
A/B/C structure highly different
grouped source prediction = 97.1%
        ↓
Hypothesis:
structural shortcuts hurt OOD
        ↓
Ablation experiment
        ↓
REJECT / NOT SUPPORTED
Removing structure hurts OOD
        ↓
Structural information is useful
despite encoding domain
        ↓
NEXT:
test lexical normalization
```

That's a good scientific progression: you formed a plausible hypothesis and your experiment **didn't support it**. You don't need to force the data to fit the original hypothesis.

The next thing I'd implement is the **raw vs light-normalized vs tool-normalized LinearSVC experiment**, keeping `all_structural` constant.


In [30]:
def build_failure_annotation_dataset(*datasets):
    frames = []

    for source, df in datasets:
        tmp = df.copy()

        # Only ground-truth failures with an annotation
        tmp = tmp[
            (tmp["label"] == -1)
            & tmp["reason"].notna()
        ].copy()

        frames.append(
            tmp[
                [
                    "group_id",
                    "trajectory_index",
                    "message_index",
                    "current_role",
                    "reason",
                    "context_text",
                    "current_text",
                ]
            ].assign(dataset=source)
        )

    result = pd.concat(
        frames,
        ignore_index=True,
    )

    return result


failure_df = build_failure_annotation_dataset(
    ("A", context_a),
    ("B", context_b),
    ("C", context_c),
)

print("Annotated failures:", len(failure_df))

print(
    failure_df["dataset"]
    .value_counts()
)

Annotated failures: 1859
dataset
B    1110
C     570
A     179
Name: count, dtype: int64


In [31]:
failure_df["reason_original"] = (
    failure_df["reason"]
    .astype(str)
)

In [32]:
import re


def clean_annotation_reason(reason):
    reason = str(reason)

    # Remove annotation prefix, but preserve actual explanation
    reason = re.sub(
        r"^\s*annotated\s+-1\s*:\s*",
        "",
        reason,
        flags=re.IGNORECASE,
    )

    reason = re.sub(
        r"\s+",
        " ",
        reason,
    ).strip()

    return reason


failure_df["reason_clean"] = (
    failure_df["reason_original"]
    .apply(clean_annotation_reason)
)

In [33]:
failure_df[
    [
        "reason_original",
        "reason_clean",
    ]
].head()

,reason_original,reason_clean
0,Annotated -1: Gives an answer that violates th...,Gives an answer that violates the 'founded in ...
1,"Annotated -1: Concludes Adelaide, but Adelaide...","Concludes Adelaide, but Adelaide was not found..."
2,"Annotated -1: Gives Adelaide, which conflicts ...","Gives Adelaide, which conflicts with the 'foun..."
3,Annotated -1: Incorrect tool call format viola...,Incorrect tool call format violating instructi...
4,Annotated -1: The submitted answer 'No. Freako...,The submitted answer 'No. Freakonomics (2010) ...


In [35]:
annotation_stats = (
    pd.concat([
        context_a.assign(dataset="A"),
        context_b.assign(dataset="B"),
        context_c.assign(dataset="C"),
    ], ignore_index=True)
    .groupby(["dataset", "label"])
    .agg(
        rows=("label", "size"),
        annotated=("reason", lambda x: x.notna().sum()),
    )
    .reset_index()
)

annotation_stats["annotation_pct"] = (
    annotation_stats["annotated"]
    / annotation_stats["rows"]
    * 100
)

print(annotation_stats)

  dataset  label  rows  annotated  annotation_pct
0       A     -1   179        179           100.0
1       A      0    56         56           100.0
2       A      1   499        499           100.0
3       B     -1  1110       1110           100.0
4       B      0   127        127           100.0
5       B      1  2320       2320           100.0
6       C     -1   570        570           100.0
7       C      0   104        104           100.0
8       C      1  1916       1916           100.0


In [36]:
all_df = pd.concat([
    context_a.assign(dataset="A"),
    context_b.assign(dataset="B"),
    context_c.assign(dataset="C"),
], ignore_index=True)

for label in [-1, 0, 1]:
    print("\n" + "=" * 80)
    print(f"LABEL {label}")
    print("=" * 80)

    examples = (
        all_df[
            (all_df["label"] == label)
            & all_df["reason"].notna()
        ][
            [
                "dataset",
                "current_role",
                "reason",
            ]
        ]
        .sample(
            n=min(
                10,
                len(
                    all_df[
                        (all_df["label"] == label)
                        & all_df["reason"].notna()
                    ]
                ),
            ),
            random_state=42,
        )
    )

    for _, row in examples.iterrows():
        print(
            f"\n[{row['dataset']}] "
            f"{row['current_role']}"
        )
        print(row["reason"])


LABEL -1

[B] ASSISTANT
Annotated -1: this step responds '{ "message": "Here are the earliest available connected Economy itineraries that meet your arrival-before-7:00 a.m. constraint where possible:\\n\\nOutbound (DTW → LGA, 2024-05-17):\\n- No one-stop options arrive before 7:00 a.m. The earliest available one-stops arrive after noon vi' and does not correct the prior failure: calls ['search_onestop_flight'], which are not among the reference workflow actions ['get_reservation_details', 'update_reservation_baggages', 'update_reservation_flights'].

[B] ASSISTANT
Annotated -1: Outputs options/pricing but depends on an ineligible cancellation; also asserts time-window fit without needed data. [review: GPT-5.2-Thinking (medium)]

[B] TOOL_CALL
Annotated -1: Continues irrelevant/redundant tool calls; no progress toward APN reset or resolution. [review: GPT-5.2-Thinking (medium)]

[B] TOOL_CALL
Annotated -1: Repeats tool misuse by calling another unavailable tool. [review: GPT-5.2]

[B]

In [37]:
def build_annotation_dataset(*datasets):
    frames = []

    for source, df in datasets:
        tmp = df.copy()

        # Keep anything that actually has an annotation.
        tmp = tmp[
            tmp["reason"].notna()
        ].copy()

        tmp["dataset"] = source

        columns = [
            "dataset",
            "trajectory_index",
            "message_index",
            "current_role",
            "label",
            "reason",
            "context_text",
            "current_text",
        ]

        # group_id if available
        if "group_id" in tmp.columns:
            columns.insert(1, "group_id")

        frames.append(
            tmp[columns]
        )

    return pd.concat(
        frames,
        ignore_index=True,
    )


annotation_df = build_annotation_dataset(
    ("A", context_a),
    ("B", context_b),
    ("C", context_c),
)

print(
    "Total annotated rows:",
    len(annotation_df)
)

print("\nLabels:")
print(
    annotation_df["label"]
    .value_counts()
    .sort_index()
)

print("\nDatasets:")
print(
    annotation_df["dataset"]
    .value_counts()
)

Total annotated rows: 6881

Labels:
label
-1    1859
 0     287
 1    4735
Name: count, dtype: int64

Datasets:
dataset
B    3557
C    2590
A     734
Name: count, dtype: int64


In [ ]:
annotation_df["reason_original"] = (
    annotation_df["reason"].astype(str)
)

In [61]:
import re


def clean_reason_semantic(reason):
    """
    Remove annotation infrastructure/metadata while preserving
    the actual human-written semantic explanation.
    """

    if reason is None:
        return ""

    reason = str(reason)

    # ---------------------------------------------------------
    # 1. Remove annotation prefix
    #
    # Annotated -1:
    # Annotated +1:
    # Annotated 0:
    # ---------------------------------------------------------

    reason = re.sub(
        r"^\s*Annotated\s*[+-]?[01]\s*:\s*",
        "",
        reason,
        flags=re.IGNORECASE,
    )

    # ---------------------------------------------------------
    # 2. Remove reviewer metadata
    #
    # Examples:
    # [review: GPT-5.2-Thinking (medium)]
    # [review: GPT-5.2]
    # [review: Qwen3]
    # [review: DeepSeek V3 Thinking]
    # ---------------------------------------------------------

    reason = re.sub(
        r"\[\s*review\s*:[^\]]*\]",
        "",
        reason,
        flags=re.IGNORECASE,
    )

    # ---------------------------------------------------------
    # 3. Normalize whitespace
    # ---------------------------------------------------------

    reason = re.sub(
        r"\s+",
        " ",
        reason,
    ).strip()

    return reason


annotation_df["reason_clean"] = (
    annotation_df["reason_original"]
    .apply(clean_annotation_reason)
)

In [62]:
annotation_df[
    [
        "label",
        "reason_original",
        "reason_clean",
    ]
].sample(
    20,
    random_state=42,
)

,label,reason_original,reason_clean
5728,1,Annotated +1: this assistant step is correct a...,this assistant step is correct and advances th...
5345,1,Annotated +1: this assistant step is correct a...,this assistant step is correct and advances th...
2906,1,Annotated +1: this assistant step is correct a...,this assistant step is correct and advances th...
3257,-1,Annotated -1: Continues the incorrect suspensi...,Continues the incorrect suspension/billing pat...
4607,0,Annotated 0: this is neutral or exploratory ra...,this is neutral or exploratory rather than cle...
5269,1,Annotated +1: this assistant step is correct a...,this assistant step is correct and advances th...
5787,1,Annotated +1: this assistant step is correct a...,this assistant step is correct and advances th...
5938,1,Annotated +1: this assistant step is correct a...,this assistant step is correct and advances th...
1498,-1,Annotated -1: Calculates totals based on incor...,Calculates totals based on incorrect baggage-f...
4184,-1,Annotated -1: Continues irrelevant/redundant t...,Continues irrelevant/redundant tool calls; no ...


In [63]:
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans

negative_annotations = annotation_df[
    annotation_df["label"] == -1
].copy()

negative_annotations = negative_annotations[
    negative_annotations["reason_clean"].notna()
].reset_index(drop=True)

reasons = (
    negative_annotations["reason_clean"]
    .astype(str)
    .tolist()
)

print("Negative annotations:", len(reasons))

reason_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

reason_embeddings = reason_model.encode(
    reasons,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
)

print(reason_embeddings.shape)

Negative annotations: 1859


Batches:   0%|          | 0/30 [00:00<?, ?it/s]

(1859, 384)


In [64]:
N_CLUSTERS = 15

cluster_model = KMeans(
    n_clusters=N_CLUSTERS,
    random_state=42,
    n_init=20,
)

negative_annotations["reason_cluster"] = (
    cluster_model.fit_predict(
        reason_embeddings
    )
)

In [65]:
print(
    negative_annotations[
        "reason_cluster"
    ].value_counts().sort_index()
)

reason_cluster
0      38
1     108
2     182
3     211
4      84
5     268
6      70
7      91
8      32
9     139
10    115
11    118
12    196
13    132
14     75
Name: count, dtype: int64


In [66]:
print(
    negative_annotations.columns.tolist()
)

['dataset', 'group_id', 'trajectory_index', 'message_index', 'current_role', 'label', 'reason', 'context_text', 'current_text', 'reason_original', 'reason_clean', 'reason_semantic', 'reason_cluster']


In [69]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
import pandas as pd


def extract_cluster_keywords(
    df,
    text_col="reason_clean",
    cluster_col="reason_cluster",
    top_n=10,
):
    results = []

    for cluster_id in sorted(df[cluster_col].unique()):

        cluster_texts = (
            df.loc[
                df[cluster_col] == cluster_id,
                text_col,
            ]
            .dropna()
            .astype(str)
            .tolist()
        )

        if len(cluster_texts) < 2:
            continue

        vectorizer = TfidfVectorizer(
            stop_words="english",
            ngram_range=(1, 3),
            min_df=2,
            max_df=0.9,
            sublinear_tf=True,
        )

        X = vectorizer.fit_transform(
            cluster_texts
        )

        terms = np.array(
            vectorizer.get_feature_names_out()
        )

        mean_scores = np.asarray(
            X.mean(axis=0)
        ).ravel()

        top_indices = mean_scores.argsort()[
            ::-1
        ][:top_n]

        top_terms = terms[top_indices]

        results.append({
            "cluster": cluster_id,
            "count": len(cluster_texts),
            "keywords": list(top_terms),
        })

    return pd.DataFrame(results)

taxonomy_candidates = extract_cluster_keywords(
    negative_annotations,
    text_col="reason_semantic",
    cluster_col="reason_cluster",
    top_n=10,
)

taxonomy_candidates

,cluster,count,keywords
0,0,38,"[sim, line, phone, step, active, check, status..."
1,1,108,"[projects, workspace, directory, wrong, cd, fi..."
2,2,182,"[tool, instead, incorrectly, search, order, ea..."
3,3,211,"[unavailable, unavailable tool, progress, call..."
4,4,84,"[tool, tool calls, continues, tool calls progr..."
5,5,268,"[tool, check, line, user, incorrect, continues..."
6,6,70,"[workflow, reference workflow, reference, call..."
7,7,91,"[required, travel_cost, book_flight, parameter..."
8,8,32,"[error, tool, incorrect, schema, correct, user..."
9,9,139,"[incorrect, answer, assistant, incorrectly, er..."


In [70]:
annotation_df["reason_semantic"] = (
    annotation_df["reason_original"]
    .apply(clean_reason_semantic)
)

contamination_terms = [
    "review:",
    "gpt",
    "qwen",
    "deepseek",
    "thinking",
]

for term in contamination_terms:

    count = (
        annotation_df["reason_semantic"]
        .str.contains(
            term,
            case=False,
            na=False,
        )
        .sum()
    )

    print(
        f"{term:12s}: {count}"
    )

review:     : 0
gpt         : 0
qwen        : 0
deepseek    : 0
thinking    : 0


In [72]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np


def show_cluster_examples(
    df,
    embeddings,
    cluster_model,
    n_examples=8,
):
    for cluster_id in range(cluster_model.n_clusters):

        indices = np.where(
            df["reason_cluster"].values == cluster_id
        )[0]

        cluster_embeddings = embeddings[indices]

        centroid = (
            cluster_model.cluster_centers_[cluster_id]
            .reshape(1, -1)
        )

        similarities = cosine_similarity(
            cluster_embeddings,
            centroid,
        ).ravel()

        best_local = np.argsort(
            similarities
        )[::-1][:n_examples]

        best_indices = indices[best_local]

        print("\n" + "=" * 100)
        print(
            f"CLUSTER {cluster_id} "
            f"(n={len(indices)})"
        )
        print("=" * 100)

        for idx in best_indices:
            row = df.iloc[idx]

            print(
                f"\n[{row['dataset']}] "
                f"{row['reason_semantic']}"
            )

show_cluster_examples(
    negative_annotations,
    reason_embeddings,
    cluster_model,
    n_examples=8,
)


CLUSTER 0 (n=38)

[B] this step responds 'Thanks — SIM looks fine and is active (no SIM lock). Next steps: 1) Mobile data toggle - Go to Settings > Mobile Network (or Connections) and turn Mobile/Data ON. - Tell me whether you were able to turn it ON and whether the "Data Disabled" icon cleared. 2) Network mode / selecti' and does not correct the prior failure: the assistant responds 'Thanks — Airplane Mode is off. Next, please check the SIM: 1) Power off the phone. 2) If your phone uses a physical SIM: remove the SIM tray, inspect the SIM for visible damage or dirt, reinsert the SIM firmly, then power the phone back on. - If your phone uses eSIM, tell me “eSI'; this claim or response conflicts with the available trajectory evidence or a required task constraint.

[B] this step responds 'Okay — please try this on your phone: 1. Take the phone out of your pocket and unlock it. 2. Open the quick settings / control center (swipe down from the top on most phones). 3. Look for the airplane

In [73]:
cluster_review = (
    negative_annotations
    .groupby("reason_cluster")
    .agg(
        count=("reason_cluster", "size"),
        datasets_present=("dataset", "nunique"),
    )
    .reset_index()
)

cluster_review["failure_family"] = pd.NA
cluster_review["failure_type"] = pd.NA
cluster_review["notes"] = pd.NA

cluster_review

,reason_cluster,count,datasets_present,failure_family,failure_type,notes
0,0,38,1,<NA>,<NA>,<NA>
1,1,108,1,<NA>,<NA>,<NA>
2,2,182,3,<NA>,<NA>,<NA>
3,3,211,3,<NA>,<NA>,<NA>
4,4,84,1,<NA>,<NA>,<NA>
5,5,268,3,<NA>,<NA>,<NA>
6,6,70,3,<NA>,<NA>,<NA>
7,7,91,2,<NA>,<NA>,<NA>
8,8,32,3,<NA>,<NA>,<NA>
9,9,139,3,<NA>,<NA>,<NA>


In [75]:
def get_cluster_representatives(
    df,
    embeddings,
    cluster_model,
    text_col="reason_semantic",
    cluster_col="reason_cluster",
    n_examples=8,
):
    rows = []

    for cluster_id in range(cluster_model.n_clusters):

        indices = np.where(
            df[cluster_col].values == cluster_id
        )[0]

        cluster_embeddings = embeddings[indices]

        centroid = (
            cluster_model.cluster_centers_[cluster_id]
            .reshape(1, -1)
        )

        similarities = cosine_similarity(
            cluster_embeddings,
            centroid,
        ).ravel()

        best_local = np.argsort(
            similarities
        )[::-1][:n_examples]

        best_indices = indices[best_local]

        examples = (
            df.iloc[best_indices][text_col]
            .astype(str)
            .tolist()
        )

        rows.append({
            "cluster": cluster_id,
            "count": len(indices),
            "representative_examples": examples,
        })

    return pd.DataFrame(rows)

In [76]:
cluster_representatives = get_cluster_representatives(
    negative_annotations,
    reason_embeddings,
    cluster_model,
)

cluster_summary = taxonomy_candidates.merge(
    cluster_representatives,
    on=["cluster", "count"],
    how="left",
)

In [90]:
FAILURE_CONCEPT_PATTERNS = {
    "repeated_action": [
        r"\brepeat",
        r"\bredundan",
        r"\bagain\b",
        r"\bwithout progress\b",
        r"\brepetition\b",
    ],

    "unavailable_tool": [
        r"\bunavailable tool",
        r"\bnon[- ]existent tool",
        r"\bnonexistent tool",
        r"\btool .* does not exist",
        r"\bmissing tool",
    ],

    "wrong_tool_or_action": [
        r"\bwrong tool",
        r"\bincorrect tool",
        r"\btool misuse",
        r"\birrelevant tool",
        r"\bwrong action",
    ],

    "irrelevant_action": [
        r"\birrelevant",
        r"\bnot relevant",
        r"\bunnecessary",
        r"\bdoes not advance",
        r"\bno progress",
    ],

    "wrong_argument": [
        r"\bwrong argument",
        r"\bincorrect argument",
        r"\binvalid argument",
        r"\binvalid path",
        r"\bwrong .* value",
        r"\bincorrect .* value",
        r"\bempty .* field",
    ],

    "missing_required_argument": [
        r"\brequired parameter",
        r"\bmissing parameter",
        r"\bmissing required",
        r"\bomits? .* parameter",
        r"\bempty .* parameter",
    ],

    "hallucinated_or_unsupported_value": [
        r"\binvent",
        r"\bfabricat",
        r"\bnot provided",
        r"\bnot supplied",
        r"\bunsupported value",
        r"\bmade[- ]?up",
    ],

    "unsupported_claim": [
        r"\bunsupported claim",
        r"\bwithout evidence",
        r"\bnot supported",
        r"\bconflicts with .* evidence",
        r"\bdespite .* showing",
        r"\basserts? .* despite",
    ],

    "incorrect_state_claim": [
        r"\bincorrectly asserts",
        r"\bincorrectly states",
        r"\bstates .* despite",
        r"\bclaims .* despite",
    ],

    "false_success_or_completion": [
        r"\breports? successful",
        r"\bclaims? success",
        r"\bconfirms? completion",
        r"\bclaims? completion",
        r"\breports? completion",
    ],

    "constraint_or_policy_violation": [
        r"\bviolat",
        r"\bpolicy",
        r"\bconstraint",
        r"\bnot allowed",
        r"\bnot requested",
        r"\bunrequested",
    ],

    "missing_required_action": [
        r"\bfails? to",
        r"\bomits?",
        r"\bmissing required action",
        r"\bdoes not .* required",
        r"\bonly drafts .* without posting",
    ],

    "unresolved_prior_error": [
        r"\bdoes not correct the prior failure",
        r"\buncorrected",
        r"\bfails? to recover",
        r"\bcontinues after .* failure",
        r"\bearlier incorrect",
        r"\bcumulative",
    ],

    "incorrect_reasoning": [
        r"\bincorrect reasoning",
        r"\bincorrect conclusion",
        r"\bconcludes .* but",
        r"\bcalculation",
        r"\bcalculated incorrectly",
    ],

    "tool_result_misinterpretation": [
        r"\bmisinterpret",
        r"\bmisread",
        r"\bdespite tool",
        r"\btool .* shows",
        r"\btool output .* but",
        r"\bresult .* but",
    ],

    "schema_or_format_error": [
        r"\bschema",
        r"\bmalformed",
        r"\bincorrect .* format",
        r"\btool call format",
        r"\binvalid format",
    ],

    "workflow_violation": [
        r"\breference workflow",
        r"\bworkflow action",
        r"\bworkflow requires",
        r"\bnot among .* workflow",
        r"\bdeviates? from .* workflow",
    ],
}

In [91]:
import re

from collections import Counter
from sklearn.metrics.pairwise import cosine_similarity

def score_failure_concepts(texts):
    scores = Counter()

    for text in texts:
        text = str(text).lower()

        for concept, patterns in FAILURE_CONCEPT_PATTERNS.items():

            if any(
                re.search(pattern, text)
                for pattern in patterns
            ):
                scores[concept] += 1

    return scores

In [92]:
def infer_cluster_taxonomy(cluster_summary):
    rows = []

    for _, row in cluster_summary.iterrows():

        examples = row["representative_examples"]

        scores = score_failure_concepts(
            examples
        )

        ranked = scores.most_common()

        if ranked:
            candidate = ranked[0][0]
            candidate_score = ranked[0][1] / len(examples)
        else:
            candidate = "unknown"
            candidate_score = 0.0

        rows.append({
            "cluster": row["cluster"],
            "count": row["count"],
            "candidate_failure_type": candidate,
            "confidence": candidate_score,
            "all_matches": ranked,
            "keywords": row["keywords"],
            "representative_examples": examples,
        })

    return pd.DataFrame(rows)

In [93]:
auto_taxonomy = infer_cluster_taxonomy(
    cluster_summary
)

auto_taxonomy[
    [
        "cluster",
        "count",
        "candidate_failure_type",
        "confidence",
        "all_matches",
    ]
]

,cluster,count,candidate_failure_type,confidence,all_matches
0,0,38,unresolved_prior_error,0.750,"[(unresolved_prior_error, 6), (unsupported_cla..."
1,1,108,repeated_action,0.125,"[(repeated_action, 1)]"
2,2,182,constraint_or_policy_violation,0.375,"[(constraint_or_policy_violation, 3)]"
3,3,211,repeated_action,0.750,"[(repeated_action, 6), (unavailable_tool, 5), ..."
4,4,84,repeated_action,1.000,"[(repeated_action, 8), (irrelevant_action, 8)]"
5,5,268,constraint_or_policy_violation,0.250,"[(constraint_or_policy_violation, 2), (repeate..."
6,6,70,workflow_violation,1.000,"[(workflow_violation, 8)]"
7,7,91,missing_required_argument,1.000,"[(missing_required_argument, 8), (missing_requ..."
8,8,32,schema_or_format_error,0.750,"[(schema_or_format_error, 6), (constraint_or_p..."
9,9,139,unresolved_prior_error,0.375,"[(unresolved_prior_error, 3), (repeated_action..."


In [94]:
failure_taxonomy = (
    auto_taxonomy
    .groupby("candidate_failure_type")
    .agg(
        clusters=("cluster", list),
        examples_count=("count", "sum"),
        mean_confidence=("confidence", "mean"),
    )
    .reset_index()
    .sort_values(
        "examples_count",
        ascending=False,
    )
)

failure_taxonomy

,candidate_failure_type,clusters,examples_count,mean_confidence
0,constraint_or_policy_violation,"[2, 5, 12, 13]",778,0.37500
4,repeated_action,"[1, 3, 4, 10]",518,0.71875
6,unresolved_prior_error,"[0, 9]",177,0.56250
1,incorrect_reasoning,[11],118,0.50000
3,missing_required_argument,[7],91,1.00000
2,missing_required_action,[14],75,0.12500
7,workflow_violation,[6],70,1.00000
5,schema_or_format_error,[8],32,0.75000


In [95]:
failure_taxonomy = (
    auto_taxonomy
    .groupby("candidate_failure_type")
    .agg(
        clusters=("cluster", list),
        examples_count=("count", "sum"),
        mean_confidence=("confidence", "mean"),
    )
    .reset_index()
    .sort_values(
        "examples_count",
        ascending=False,
    )
)

failure_taxonomy

,candidate_failure_type,clusters,examples_count,mean_confidence
0,constraint_or_policy_violation,"[2, 5, 12, 13]",778,0.37500
4,repeated_action,"[1, 3, 4, 10]",518,0.71875
6,unresolved_prior_error,"[0, 9]",177,0.56250
1,incorrect_reasoning,[11],118,0.50000
3,missing_required_argument,[7],91,1.00000
2,missing_required_action,[14],75,0.12500
7,workflow_violation,[6],70,1.00000
5,schema_or_format_error,[8],32,0.75000


In [96]:
cluster_to_type = dict(
    zip(
        auto_taxonomy["cluster"],
        auto_taxonomy["candidate_failure_type"],
    )
)

negative_annotations["failure_type_auto"] = (
    negative_annotations["reason_cluster"]
    .map(cluster_to_type)
)

taxonomy_by_dataset = pd.crosstab(
    negative_annotations["failure_type_auto"],
    negative_annotations["dataset"],
)

taxonomy_by_dataset

dataset,A,B,C
failure_type_auto,,,
constraint_or_policy_violation,65,498,215
incorrect_reasoning,3,38,77
missing_required_action,10,37,28
missing_required_argument,0,43,48
repeated_action,23,367,128
schema_or_format_error,16,10,6
unresolved_prior_error,56,90,31
workflow_violation,6,27,37


In [97]:
negative_annotations[
    [
        "dataset",
        "reason_semantic",
        "reason_cluster",
        "failure_type_auto",
    ]
].sample(
    30,
    random_state=42,
)

,dataset,reason_semantic,reason_cluster,failure_type_auto
233,B,"this step responds '{ ""message"": ""Here are the...",6,workflow_violation
450,B,Outputs options/pricing but depends on an inel...,12,constraint_or_policy_violation
1202,B,Continues irrelevant/redundant tool calls; no ...,4,repeated_action
1244,B,Repeats tool misuse by calling another unavail...,3,repeated_action
411,B,Another status check that still doesn’t resolv...,13,constraint_or_policy_violation
1339,C,cd .. continues workflow with the wrong folder...,1,repeated_action
1655,C,Confirms ticket creation but adds unrequested ...,13,constraint_or_policy_violation
1225,B,Continues irrelevant/redundant tool calls; no ...,4,repeated_action
1170,B,Incorrectly asserts the line is active and eve...,5,constraint_or_policy_violation
1169,B,Gives generic troubleshooting but fails to pri...,5,constraint_or_policy_violation


In [100]:
def classify_reason_multilabel(reason):
    text = str(reason).lower()

    matches = []

    for failure_type, patterns in FAILURE_CONCEPT_PATTERNS.items():

        matched_patterns = [
            pattern
            for pattern in patterns
            if re.search(pattern, text)
        ]

        if matched_patterns:
            matches.append({
                "failure_type": failure_type,
                "score": len(matched_patterns),
                "matched_patterns": matched_patterns,
            })

    return sorted(
        matches,
        key=lambda x: x["score"],
        reverse=True,
    )

In [101]:
negative_annotations["taxonomy_matches"] = (
    negative_annotations["reason_semantic"]
    .apply(classify_reason_multilabel)
)

In [102]:
negative_annotations["failure_types"] = (
    negative_annotations["taxonomy_matches"]
    .apply(
        lambda matches: [
            item["failure_type"]
            for item in matches
        ]
    )
)

In [103]:
def get_primary_failure(matches):
    if not matches:
        return "unknown"

    return matches[0]["failure_type"]


negative_annotations["failure_type"] = (
    negative_annotations["taxonomy_matches"]
    .apply(get_primary_failure)
)

In [104]:
negative_annotations[
    [
        "dataset",
        "reason_semantic",
        "reason_cluster",
        "failure_type",
        "failure_types",
    ]
].sample(
    50,
    random_state=42,
)

,dataset,reason_semantic,reason_cluster,failure_type,failure_types
233,B,"this step responds '{ ""message"": ""Here are the...",6,workflow_violation,"[workflow_violation, constraint_or_policy_viol..."
450,B,Outputs options/pricing but depends on an inel...,12,unknown,[]
1202,B,Continues irrelevant/redundant tool calls; no ...,4,irrelevant_action,"[irrelevant_action, repeated_action]"
1244,B,Repeats tool misuse by calling another unavail...,3,repeated_action,"[repeated_action, unavailable_tool, wrong_tool..."
411,B,Another status check that still doesn’t resolv...,13,unknown,[]
1339,C,cd .. continues workflow with the wrong folder...,1,unknown,[]
1655,C,Confirms ticket creation but adds unrequested ...,13,constraint_or_policy_violation,"[constraint_or_policy_violation, unresolved_pr..."
1225,B,Continues irrelevant/redundant tool calls; no ...,4,irrelevant_action,"[irrelevant_action, repeated_action]"
1170,B,Incorrectly asserts the line is active and eve...,5,unsupported_claim,"[unsupported_claim, incorrect_state_claim, too..."
1169,B,Gives generic troubleshooting but fails to pri...,5,missing_required_action,[missing_required_action]


In [105]:
taxonomy_counts = (
    negative_annotations["failure_type"]
    .value_counts(dropna=False)
    .rename_axis("failure_type")
    .reset_index(name="count")
)

taxonomy_counts["percentage"] = (
    taxonomy_counts["count"]
    / len(negative_annotations)
    * 100
)

taxonomy_counts

,failure_type,count,percentage
0,unknown,742,39.913932
1,repeated_action,331,17.805272
2,constraint_or_policy_violation,227,12.210866
3,irrelevant_action,113,6.078537
4,missing_required_argument,62,3.335126
5,unresolved_prior_error,61,3.281334
6,missing_required_action,48,2.582033
7,unsupported_claim,47,2.528241
8,unavailable_tool,47,2.528241
9,workflow_violation,46,2.474449


In [106]:
unknown_df = negative_annotations[
    negative_annotations["failure_type"]
    == "unknown"
].copy()

print(
    f"Unknown: {len(unknown_df):,} "
    f"({len(unknown_df) / len(negative_annotations) * 100:.2f}%)"
)

Unknown: 742 (39.91%)


In [107]:
unknown_df[
    [
        "dataset",
        "reason_semantic",
        "reason_cluster",
    ]
].sample(
    n=min(100, len(unknown_df)),
    random_state=42,
)

,dataset,reason_semantic,reason_cluster
271,B,Correctly notes cabins can’t differ within one...,13
439,B,Offers flight options for a disallowed origin/...,7
837,B,The assistant sends a transfer message without...,9
1483,C,The assistant failed to search for the file as...,14
1001,B,The assistant correctly transfers the user to ...,9
...,...,...,...
1699,C,"Claims the limit is 20,000 RMB, but the budget...",11
5,A,"Incorrectly attributes ""See Yourself"" to Roger...",9
888,B,User requested check_status_bar but assistant ...,5
1695,C,The assistant failed to follow the guidelines ...,9


In [108]:
unknown_reasons = (
    unknown_df["reason_semantic"]
    .fillna("")
    .astype(str)
    .tolist()
)

In [109]:
unknown_embeddings = reason_model.encode(
    unknown_reasons,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
)

Batches:   0%|          | 0/12 [00:00<?, ?it/s]

In [110]:
from sklearn.cluster import KMeans

UNKNOWN_CLUSTERS = 10

unknown_cluster_model = KMeans(
    n_clusters=UNKNOWN_CLUSTERS,
    random_state=42,
    n_init=20,
)

unknown_df["unknown_cluster"] = (
    unknown_cluster_model.fit_predict(
        unknown_embeddings
    )
)

In [112]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np


def show_unknown_cluster_examples(
    df,
    embeddings,
    cluster_model,
    n_examples=10,
):
    for cluster_id in range(cluster_model.n_clusters):

        indices = np.where(
            df["unknown_cluster"].values == cluster_id
        )[0]

        cluster_embeddings = embeddings[indices]

        centroid = (
            cluster_model.cluster_centers_[cluster_id]
            .reshape(1, -1)
        )

        similarities = cosine_similarity(
            cluster_embeddings,
            centroid,
        ).ravel()

        best_local = np.argsort(
            similarities
        )[::-1][:n_examples]

        best_indices = indices[best_local]

        print("\n" + "=" * 100)
        print(
            f"UNKNOWN CLUSTER {cluster_id} "
            f"(n={len(indices)})"
        )
        print("=" * 100)

        for idx in best_indices:
            row = df.iloc[idx]

            print(
                f"\n[{row['dataset']}] "
                f"{row['reason_semantic']}"
            )


show_unknown_cluster_examples(
    unknown_df,
    unknown_embeddings,
    unknown_cluster_model,
    n_examples=10,
)


UNKNOWN CLUSTER 0 (n=56)

[C] Incorrectly claims Projects was created in the workspace folder. It reinforces the earlier location error.

[C] Created Projects in the wrong place (under alex/, not workspace/) because it never cd'ed into workspace.

[C] Moves to destination 'Projects' within workspace, which is not the intended directory; it results in an incorrect file placement. This further corrupts the workspace state.

[C] Tries to create Projects in workspace but hits a conflict caused by earlier wrong move. Still no cleanup or correct setup.

[C] Created Projects in the current (top-level) directory instead of inside workspace as requested.

[C] Lists files in misplaced Projects; confirms creation but not in workspace/Projects.

[C] Incorrectly claimed Projects was created in the workspace folder; state shows it was not.

[C] Changes into /alex/Projects, which is not the requested workspace/Projects directory. Continues operating in the wrong place.

[C] Returns to workspace, but

In [113]:
unknown_keywords = extract_cluster_keywords(
    unknown_df,
    text_col="reason_semantic",
    cluster_col="unknown_cluster",
    top_n=12,
)

unknown_keywords

,cluster,count,keywords
0,0,56,"[workspace, projects, directory, workspace pro..."
1,1,127,"[booking, incorrect, tool, flight, baggage, pa..."
2,2,104,"[line, tool, diagnostic, continues, device, ch..."
3,3,130,"[incorrect, continues, transfer, correct, line..."
4,4,23,"[fuel, tank, incorrectly, gallons, fillfueltan..."
5,5,26,"[miles, distance, tool, feasibility, km, unit,..."
6,6,106,"[answer, search, incorrect, incorrectly, does,..."
7,7,76,"[tool, invalid, error, invalid tool, incorrect..."
8,8,56,"[file, directory, wrong, content, incorrectly,..."
9,9,38,"[tweet, user, incorrectly, despite, message, s..."


In [114]:
unknown_summary = (
    unknown_df
    .groupby("unknown_cluster")
    .agg(
        count=("unknown_cluster", "size"),
        datasets_present=("dataset", "nunique"),
    )
    .reset_index()
    .merge(
        unknown_keywords,
        left_on=["unknown_cluster", "count"],
        right_on=["cluster", "count"],
        how="left",
    )
    .drop(columns="cluster")
)

unknown_summary

,unknown_cluster,count,datasets_present,keywords
0,0,56,1,"[workspace, projects, directory, workspace pro..."
1,1,127,2,"[booking, incorrect, tool, flight, baggage, pa..."
2,2,104,3,"[line, tool, diagnostic, continues, device, ch..."
3,3,130,3,"[incorrect, continues, transfer, correct, line..."
4,4,23,1,"[fuel, tank, incorrectly, gallons, fillfueltan..."
5,5,26,1,"[miles, distance, tool, feasibility, km, unit,..."
6,6,106,3,"[answer, search, incorrect, incorrectly, does,..."
7,7,76,3,"[tool, invalid, error, invalid tool, incorrect..."
8,8,56,2,"[file, directory, wrong, content, incorrectly,..."
9,9,38,2,"[tweet, user, incorrectly, despite, message, s..."


In [121]:
NEW_FAILURE_PATTERNS = {

    # -----------------------------------------------------
    # Operation applied at the wrong place/state/location
    # -----------------------------------------------------

    "incorrect_state_or_location": [
        r"\bwrong place\b",
        r"\bwrong location\b",
        r"\bwrong folder\b",
        r"\bwrong directory\b",
        r"\bincorrect location\b",
        r"\bincorrect file placement\b",
        r"\bnot .* intended directory\b",
        r"\binstead of inside\b",
        r"\boperating in the wrong\b",
        r"\bcreated .* instead of\b",
        r"\bmisplaced\b",
        r"\bwrong workspace\b",
        r"\bcorrupts? .* state\b",
    ],

    # -----------------------------------------------------
    # Claims current state is X when trajectory says Y
    # -----------------------------------------------------

    "state_mismatch": [
        r"\bstate shows .* not\b",
        r"\bstate shows\b",
        r"\bcontradicts? .* state\b",
        r"\bincorrect claim .* state\b",
        r"\bclaim .* despite .* state\b",
        r"\beven though .* exists\b",
        r"\beven though .* active\b",
        r"\beven though .* available\b",
    ],

    # -----------------------------------------------------
    # Wrong identifier / field used for another semantic field
    # -----------------------------------------------------

    "identifier_or_field_mismatch": [
        r"\buses? .* id as .* email\b",
        r"\buses? .* email as .* id\b",
        r"\bwrong .* id\b",
        r"\bincorrect .* id\b",
        r"\bwrong identifier\b",
        r"\bincorrect identifier\b",
        r"\buses? .* as .* identifier\b",
        r"\bfield mismatch\b",
    ],

    # -----------------------------------------------------
    # Authentication / credential misuse
    # -----------------------------------------------------

    "authentication_or_credentials_error": [
        r"\bauthentication\b",
        r"\bauthenticate\b",
        r"\blogin credential\b",
        r"\bcredentials?\b",
        r"\baccess token\b",
        r"\bwrong .* email .* authentication\b",
    ],

    # -----------------------------------------------------
    # Does something correctly locally, but based on bad state
    # -----------------------------------------------------

    "propagated_incorrect_state": [
        r"\breinforces? .* earlier .* error\b",
        r"\bcaused by earlier wrong\b",
        r"\bbased on .* earlier .* error\b",
        r"\bcontinues operating .* wrong\b",
        r"\bcontinues .* incorrect .* state\b",
        r"\breflects .* earlier .* incorrect\b",
    ],
}

In [122]:
for failure_type, patterns in NEW_FAILURE_PATTERNS.items():
    FAILURE_CONCEPT_PATTERNS.setdefault(
        failure_type,
        []
    ).extend(patterns)

In [123]:
negative_annotations["taxonomy_matches"] = (
    negative_annotations["reason_semantic"]
    .apply(classify_reason_multilabel)
)

negative_annotations["failure_types"] = (
    negative_annotations["taxonomy_matches"]
    .apply(
        lambda matches: [
            item["failure_type"]
            for item in matches
        ]
    )
)

negative_annotations["failure_type"] = (
    negative_annotations["taxonomy_matches"]
    .apply(get_primary_failure)
)

In [124]:
taxonomy_counts = (
    negative_annotations["failure_type"]
    .value_counts()
    .rename_axis("failure_type")
    .reset_index(name="count")
)

taxonomy_counts["percentage"] = (
    taxonomy_counts["count"]
    / len(negative_annotations)
    * 100
)

taxonomy_counts

,failure_type,count,percentage
0,unknown,660,35.502959
1,repeated_action,331,17.805272
2,constraint_or_policy_violation,227,12.210866
3,irrelevant_action,113,6.078537
4,missing_required_argument,62,3.335126
5,unresolved_prior_error,61,3.281334
6,missing_required_action,48,2.582033
7,unavailable_tool,47,2.528241
8,unsupported_claim,47,2.528241
9,workflow_violation,46,2.474449


In [125]:
unknown_df_v2 = negative_annotations[
    negative_annotations["failure_type"] == "unknown"
].copy()

print(
    f"Unknown v2: {len(unknown_df_v2)} "
    f"({len(unknown_df_v2) / len(negative_annotations) * 100:.2f}%)"
)

Unknown v2: 660 (35.50%)


In [128]:
def summarize_unknown_clusters(
    unknown_df,
    embeddings,
    model,
    keywords_df,
    n_examples=5,
):
    rows = []

    keyword_map = {
        row["cluster"]: row["keywords"]
        for _, row in keywords_df.iterrows()
    }

    for cluster_id in range(
        model.n_clusters
    ):

        indices = np.where(
            unknown_df["unknown_cluster"].values
            == cluster_id
        )[0]

        cluster_emb = embeddings[indices]

        centroid = (
            model.cluster_centers_[cluster_id]
            .reshape(1, -1)
        )

        similarities = cosine_similarity(
            cluster_emb,
            centroid,
        ).ravel()

        top_local = np.argsort(
            similarities
        )[::-1][:n_examples]

        top_indices = indices[
            top_local
        ]

        examples = (
            unknown_df.iloc[top_indices][
                "reason_semantic"
            ]
            .tolist()
        )

        rows.append({
            "cluster": cluster_id,
            "count": len(indices),
            "keywords": keyword_map.get(
                cluster_id,
                []
            ),
            "examples": examples,
        })

    return pd.DataFrame(rows)

In [129]:
unknown_cluster_summary = (
    summarize_unknown_clusters(
        unknown_df,
        unknown_embeddings,
        unknown_cluster_model,
        unknown_keywords,
        n_examples=5,
    )
)

unknown_cluster_summary

,cluster,count,keywords,examples
0,0,56,"[workspace, projects, directory, workspace pro...",[Incorrectly claims Projects was created in th...
1,1,127,"[booking, incorrect, tool, flight, baggage, pa...",[Confirms a booking that is incorrect due to m...
2,2,104,"[line, tool, diagnostic, continues, device, ch...",[Continues a flawed diagnostic plan (device-si...
3,3,130,"[incorrect, continues, transfer, correct, line...",[Confirms cancellation and continues workflow ...
4,4,23,"[fuel, tank, incorrectly, gallons, fillfueltan...",[Incorrectly tried to add 50 gallons (additive...
5,5,26,"[miles, distance, tool, feasibility, km, unit,...",[Used km distance as miles when calling the mi...
6,6,106,"[answer, search, incorrect, incorrectly, does,...",[Provides an unsupported final answer not just...
7,7,76,"[tool, invalid, error, invalid tool, incorrect...",[Another invalid tool call; no corrective acti...
8,8,56,"[file, directory, wrong, content, incorrectly,...",[Attempts to write to summary.txt before creat...
9,9,38,"[tweet, user, incorrectly, despite, message, s...",[Unnecessarily requested username/password eve...


In [130]:
NEW_FAILURE_PATTERNS_V3 = {

    # Cluster 0
    "incorrect_state_or_location": [
        r"\bwrong (?:place|location|directory|folder)\b",
        r"\bincorrect (?:place|location|directory|folder)\b",
        r"\bwrong workspace\b",
        r"\bmisplaced\b",
        r"\bnot (?:in|inside) .* requested\b",
        r"\binstead of .* workspace\b",
        r"\boperating in .* wrong\b",
    ],

    # Cluster 1
    "incorrect_operation_or_transaction": [
        r"\bincorrect booking\b",
        r"\bincorrect reservation\b",
        r"\bincorrect transaction\b",
        r"\bwrong booking\b",
        r"\bwrong reservation\b",
        r"\bincorrect modification\b",
        r"\bincorrectly (?:booked|cancelled|canceled|modified)\b",
    ],

    # Cluster 2
    "incorrect_procedure": [
        r"\bflawed .* plan\b",
        r"\bincorrect .* procedure\b",
        r"\bwrong .* procedure\b",
        r"\bincorrect troubleshooting\b",
        r"\bwrong troubleshooting\b",
        r"\bcontinues .* flawed\b",
        r"\binappropriate .* step\b",
    ],

    # Cluster 3
    "incorrect_workflow_transition": [
        r"\bcontinues workflow\b",
        r"\bcontinues .* workflow\b",
        r"\bincorrect .* transfer\b",
        r"\bincorrect .* cancellation\b",
        r"\bproceeds .* despite\b",
        r"\bcontinues after\b",
    ],

    # Cluster 4
    "incorrect_quantity_or_limit": [
        r"\bincorrect .* gallons?\b",
        r"\bwrong .* gallons?\b",
        r"\bincorrect .* quantity\b",
        r"\bwrong .* quantity\b",
        r"\bexceeds? .* limit\b",
        r"\bover .* limit\b",
        r"\bincorrect .* limit\b",
    ],

    # Cluster 5
    "unit_conversion_error": [
        r"\bkm .* as miles\b",
        r"\bmiles .* as km\b",
        r"\bunit conversion\b",
        r"\bwrong unit\b",
        r"\bincorrect unit\b",
        r"\bunit mismatch\b",
        r"\bwithout converting\b",
        r"\bfailed to convert\b",
    ],

    # Cluster 6
    "unsupported_or_incorrect_answer": [
        r"\bunsupported final answer\b",
        r"\bincorrect final answer\b",
        r"\bwrong final answer\b",
        r"\banswer .* not justified\b",
        r"\banswer .* unsupported\b",
        r"\bnot justified by\b",
        r"\bdoes not support .* answer\b",
    ],

    # Cluster 7
    "invalid_tool_call": [
        r"\binvalid tool call\b",
        r"\binvalid tool invocation\b",
        r"\banother invalid tool\b",
        r"\bincorrect tool call\b",
        r"\btool call .* invalid\b",
    ],

    # Cluster 8
    "missing_prerequisite_or_order_error": [
        r"\bbefore creat",
        r"\bbefore .* exists\b",
        r"\bwithout first\b",
        r"\bmissing prerequisite\b",
        r"\bprerequisite .* missing\b",
        r"\bwrong order\b",
        r"\bout of order\b",
        r"\bdoes not exist yet\b",
    ],

    # Cluster 9
    "unnecessary_information_request": [
        r"\bunnecessarily request",
        r"\bunnecessary request",
        r"\brequests? .* unnecessarily\b",
        r"\basks? for .* despite\b",
        r"\brequests? .* despite\b",
        r"\balready (?:known|available|provided)\b",
    ],
}

In [131]:
for failure_type, patterns in NEW_FAILURE_PATTERNS_V3.items():
    FAILURE_CONCEPT_PATTERNS.setdefault(
        failure_type,
        []
    ).extend(patterns)

In [132]:
negative_annotations["taxonomy_matches"] = (
    negative_annotations["reason_semantic"]
    .apply(classify_reason_multilabel)
)

negative_annotations["failure_types"] = (
    negative_annotations["taxonomy_matches"]
    .apply(
        lambda x: [
            item["failure_type"]
            for item in x
        ]
    )
)

negative_annotations["failure_type"] = (
    negative_annotations["taxonomy_matches"]
    .apply(get_primary_failure)
)

In [133]:
taxonomy_counts_v3 = (
    negative_annotations["failure_type"]
    .value_counts()
    .rename_axis("failure_type")
    .reset_index(name="count")
)

taxonomy_counts_v3["percentage"] = (
    taxonomy_counts_v3["count"]
    / len(negative_annotations)
    * 100
)

display(taxonomy_counts_v3)


unknown_df_v3 = negative_annotations[
    negative_annotations["failure_type"] == "unknown"
].copy()

print(
    f"Unknown v3: {len(unknown_df_v3):,} "
    f"({len(unknown_df_v3) / len(negative_annotations) * 100:.2f}%)"
)

,failure_type,count,percentage
0,unknown,608,32.705756
1,repeated_action,327,17.590102
2,constraint_or_policy_violation,227,12.210866
3,irrelevant_action,111,5.970952
4,missing_required_argument,62,3.335126
5,unresolved_prior_error,61,3.281334
6,missing_required_action,48,2.582033
7,unavailable_tool,47,2.528241
8,unsupported_claim,47,2.528241
9,workflow_violation,46,2.474449


Unknown v3: 608 (32.71%)


In [134]:
TAXONOMY_DESCRIPTIONS = {
    "repeated_action":
        "The agent unnecessarily repeats an action, tool call, lookup, or step without making useful progress.",

    "constraint_or_policy_violation":
        "The agent violates an explicit task constraint, policy, rule, eligibility condition, or user requirement.",

    "irrelevant_action":
        "The agent performs an action or tool call that is irrelevant to solving the current task.",

    "missing_required_argument":
        "A required argument, parameter, identifier, or field is missing from a tool call or operation.",

    "unresolved_prior_error":
        "The agent continues without correcting an earlier failure, causing the previous error to propagate.",

    "missing_required_action":
        "The agent fails to perform an action that is required to complete the task correctly.",

    "unavailable_tool":
        "The agent attempts to use a tool or capability that is unavailable or does not exist.",

    "unsupported_claim":
        "The agent makes a claim that is not supported by the available evidence, context, or tool results.",

    "workflow_violation":
        "The agent deviates from the required workflow, sequence of actions, or reference procedure.",

    "hallucinated_or_unsupported_value":
        "The agent invents, fabricates, or uses a value that was not provided or supported by evidence.",

    "incorrect_state_or_location":
        "The agent acts on or describes the wrong location, directory, workspace, record, object, or state.",

    "wrong_tool_or_action":
        "The agent selects the wrong tool or action for the task.",

    "schema_or_format_error":
        "The agent produces a malformed tool call, invalid schema, or incorrectly formatted structured output.",

    "incorrect_state_claim":
        "The agent incorrectly describes the current state despite evidence showing a different state.",

    "tool_result_misinterpretation":
        "The agent incorrectly interprets or draws the wrong conclusion from a tool result.",

    "invalid_tool_call":
        "The agent makes an invalid tool invocation that cannot be executed correctly.",

    "unit_conversion_error":
        "The agent incorrectly converts, interprets, or uses measurement units.",

    "authentication_or_credentials_error":
        "The agent incorrectly handles authentication, credentials, identity information, or authentication fields.",

    "wrong_task_or_scope":
        "The agent addresses the wrong task, goes outside the requested scope, or fails to address the actual request.",

    "incorrect_reasoning":
        "The agent reaches an incorrect conclusion because of faulty reasoning or calculation.",

    "incorrect_value":
        "The agent uses or states an incorrect numeric, categorical, or other concrete value.",

    "premature_or_unnecessary_escalation":
        "The agent unnecessarily or prematurely escalates or transfers the task instead of continuing appropriately.",

    "factual_error":
        "The agent states an incorrect fact, attribution, entity relationship, or factual answer.",

    "wrong_argument":
        "The agent supplies an incorrect value or object for an argument or parameter.",

    "missing_prerequisite_or_order_error":
        "The agent performs an operation before a required prerequisite or performs required steps in the wrong order.",

    "incorrect_workflow_transition":
        "The agent incorrectly moves the workflow into another state, stage, transfer, cancellation, or transition.",

    "incorrect_operation_or_transaction":
        "The agent performs or confirms an incorrect transaction, booking, reservation, modification, or operation.",

    "incorrect_procedure":
        "The agent follows an incorrect procedure, troubleshooting plan, or sequence of operational steps.",

    "unnecessary_information_request":
        "The agent unnecessarily asks for information that is already available or is not required.",

    "false_success_or_completion":
        "The agent incorrectly claims that an action, transaction, or task succeeded or was completed.",

    "propagated_incorrect_state":
        "The agent continues operating from an earlier incorrect state and propagates that state into later actions.",

    "state_mismatch":
        "The agent's assumed or stated state conflicts with the actual state shown by the trajectory or tools.",

    "unsupported_or_incorrect_answer":
        "The agent provides a final answer that is incorrect or not justified by the available evidence.",

    "wrong_search_or_retrieval":
        "The agent searches for or retrieves the wrong information, entity, file, route, or target.",

    "invalid_dependency_or_prerequisite":
        "The agent's action depends on an invalid, unavailable, ineligible, or unsatisfied prerequisite.",
}

In [135]:
taxonomy_names = list(
    TAXONOMY_DESCRIPTIONS.keys()
)

taxonomy_texts = [
    TAXONOMY_DESCRIPTIONS[name]
    for name in taxonomy_names
]

taxonomy_embeddings = reason_model.encode(
    taxonomy_texts,
    batch_size=32,
    normalize_embeddings=True,
    show_progress_bar=False,
)

print(taxonomy_embeddings.shape)

(35, 384)


In [136]:
reason_texts = (
    negative_annotations["reason_semantic"]
    .fillna("")
    .astype(str)
    .tolist()
)

all_reason_embeddings = reason_model.encode(
    reason_texts,
    batch_size=64,
    normalize_embeddings=True,
    show_progress_bar=True,
)

Batches:   0%|          | 0/30 [00:00<?, ?it/s]

In [137]:
similarities = (
    all_reason_embeddings
    @ taxonomy_embeddings.T
)

print(similarities.shape)

(1859, 35)


In [138]:
best_indices = similarities.argmax(axis=1)

best_scores = similarities[
    np.arange(len(similarities)),
    best_indices,
]

semantic_failure_types = [
    taxonomy_names[i]
    for i in best_indices
]

negative_annotations[
    "semantic_failure_type"
] = semantic_failure_types

negative_annotations[
    "semantic_similarity"
] = best_scores

In [139]:
negative_annotations[
    [
        "dataset",
        "reason_semantic",
        "failure_type",
        "semantic_failure_type",
        "semantic_similarity",
    ]
].sample(
    50,
    random_state=42,
)

,dataset,reason_semantic,failure_type,semantic_failure_type,semantic_similarity
233,B,"this step responds '{ ""message"": ""Here are the...",workflow_violation,incorrect_procedure,0.367032
450,B,Outputs options/pricing but depends on an inel...,invalid_dependency_or_prerequisite,unit_conversion_error,0.348058
1202,B,Continues irrelevant/redundant tool calls; no ...,irrelevant_action,repeated_action,0.393576
1244,B,Repeats tool misuse by calling another unavail...,repeated_action,unavailable_tool,0.608067
411,B,Another status check that still doesn’t resolv...,unknown,incorrect_operation_or_transaction,0.577496
1339,C,cd .. continues workflow with the wrong folder...,incorrect_state_or_location,incorrect_workflow_transition,0.378649
1655,C,Confirms ticket creation but adds unrequested ...,constraint_or_policy_violation,incorrect_operation_or_transaction,0.431032
1225,B,Continues irrelevant/redundant tool calls; no ...,irrelevant_action,repeated_action,0.393576
1170,B,Incorrectly asserts the line is active and eve...,unsupported_claim,invalid_tool_call,0.333445
1169,B,Gives generic troubleshooting but fails to pri...,missing_required_action,incorrect_procedure,0.336289


In [140]:
negative_annotations[
    "semantic_similarity"
].describe(
    percentiles=[
        .05,
        .10,
        .25,
        .50,
        .75,
        .90,
        .95,
    ]
)

count    1859.000000
mean        0.407044
std         0.105986
min         0.048982
5%          0.241052
10%         0.276642
25%         0.333265
50%         0.395657
75%         0.480505
90%         0.547457
95%         0.598694
max         0.742344
Name: semantic_similarity, dtype: float64

In [141]:
negative_annotations[
    [
        "reason_semantic",
        "semantic_failure_type",
        "semantic_similarity",
    ]
].sort_values(
    "semantic_similarity"
).head(100)

,reason_semantic,semantic_failure_type,semantic_similarity
98,The submitted answer 'Raven — Tara Strong’s ro...,wrong_task_or_scope,0.048982
102,The submitted answer 'Arab (specifically Yemen...,unsupported_or_incorrect_answer,0.078572
15,Does not provide the specific coastal area; gi...,wrong_task_or_scope,0.100711
89,"Answers with The Joe Schmo Show, which is not ...",factual_error,0.123627
18,"Incorrect answer; Firth of Forth is north, not...",unsupported_or_incorrect_answer,0.125732
...,...,...,...
1434,Misinterprets an empty archives directory as a...,missing_required_argument,0.241427
1345,"Creates notes.md, but in the wrong directory r...",incorrect_state_or_location,0.241494
1536,Incorrectly tried to add 50 gallons (additive)...,incorrect_reasoning,0.241631
159,Makes an unsupported guess that (ON)CHOH is an...,schema_or_format_error,0.241672


In [142]:
semantic_unknowns = negative_annotations[
    negative_annotations["failure_type"]
    == "unknown"
][
    [
        "dataset",
        "reason_semantic",
        "reason_cluster",
        "semantic_failure_type",
        "semantic_similarity",
    ]
].copy()

semantic_unknowns.sort_values(
    "semantic_similarity",
    ascending=False,
).head(100)

,dataset,reason_semantic,reason_cluster,semantic_failure_type,semantic_similarity
681,B,Calls a tool that is not available in the prov...,3,missing_required_argument,0.713543
1152,B,Called a tool that is not available in the too...,3,missing_required_argument,0.705747
957,B,Calls a tool that is not available in the prov...,3,missing_required_argument,0.673378
353,B,The agent failed to challenge the user's incor...,9,factual_error,0.662940
823,B,Invoked a tool not present in the provided too...,3,missing_required_argument,0.661917
...,...,...,...,...,...
273,B,Uses tools incorrectly (invalid passenger chan...,13,incorrect_operation_or_transaction,0.488384
1072,B,Attempts another unavailable diagnostic tool w...,3,unavailable_tool,0.487924
1685,C,"Summary matches tool outputs, but depends on e...",13,incorrect_operation_or_transaction,0.487538
444,B,"Performs a flight search, but it continues pur...",7,wrong_search_or_retrieval,0.487215


In [143]:
semantic_unknowns.sort_values(
    "semantic_similarity",
    ascending=True,
).head(100)

,dataset,reason_semantic,reason_cluster,semantic_failure_type,semantic_similarity
98,A,The submitted answer 'Raven — Tara Strong’s ro...,9,wrong_task_or_scope,0.048982
102,A,The submitted answer 'Arab (specifically Yemen...,9,unsupported_or_incorrect_answer,0.078572
15,A,Does not provide the specific coastal area; gi...,11,wrong_task_or_scope,0.100711
82,A,"The submitted answer ""I'm a Jayhawk"" does not ...",9,wrong_task_or_scope,0.141576
86,A,"The submitted answer ""I'm a Jayhawk"" does not ...",9,wrong_task_or_scope,0.141576
...,...,...,...,...,...
944,B,Performs device record lookup; still not the r...,5,wrong_search_or_retrieval,0.275505
1074,B,"Valid bill lookup, but overall path still depe...",12,wrong_search_or_retrieval,0.275576
204,B,Finds a cheapest itinerary and computes totals...,7,incorrect_operation_or_transaction,0.276867
1576,C,"Uses 750.0 as miles for mileage feasibility, b...",11,unit_conversion_error,0.276899


In [144]:
prototype_rows = negative_annotations[
    negative_annotations["failure_type"] != "unknown"
].copy()

prototype_rows["n_matches"] = (
    prototype_rows["failure_types"]
    .apply(len)
)

# Start with single-label examples only.
# These are cleaner prototypes than compound failures.
prototype_rows = prototype_rows[
    prototype_rows["n_matches"] == 1
].copy()

print(
    prototype_rows["failure_type"]
    .value_counts()
)

failure_type
repeated_action                        182
constraint_or_policy_violation         160
unresolved_prior_error                  54
unavailable_tool                        42
missing_required_action                 41
workflow_violation                      33
incorrect_state_or_location             27
irrelevant_action                       26
hallucinated_or_unsupported_value       24
missing_required_argument               15
wrong_tool_or_action                    15
tool_result_misinterpretation           14
incorrect_state_claim                   14
unsupported_claim                       13
unit_conversion_error                   13
invalid_tool_call                       12
wrong_task_or_scope                     10
authentication_or_credentials_error      8
schema_or_format_error                   8
incorrect_value                          6
incorrect_reasoning                      6
premature_or_unnecessary_escalation      5
missing_prerequisite_or_order_error      

In [145]:
MIN_PROTOTYPE_EXAMPLES = 5

valid_types = (
    prototype_rows["failure_type"]
    .value_counts()
)

valid_types = valid_types[
    valid_types >= MIN_PROTOTYPE_EXAMPLES
].index.tolist()

print(
    "Prototype categories:",
    len(valid_types)
)

print(valid_types)

Prototype categories: 23
['repeated_action', 'constraint_or_policy_violation', 'unresolved_prior_error', 'unavailable_tool', 'missing_required_action', 'workflow_violation', 'incorrect_state_or_location', 'irrelevant_action', 'hallucinated_or_unsupported_value', 'missing_required_argument', 'wrong_tool_or_action', 'tool_result_misinterpretation', 'incorrect_state_claim', 'unsupported_claim', 'unit_conversion_error', 'invalid_tool_call', 'wrong_task_or_scope', 'authentication_or_credentials_error', 'schema_or_format_error', 'incorrect_value', 'incorrect_reasoning', 'premature_or_unnecessary_escalation', 'missing_prerequisite_or_order_error']


In [146]:
prototype_embeddings = {}

for failure_type in valid_types:

    texts = (
        prototype_rows.loc[
            prototype_rows["failure_type"] == failure_type,
            "reason_semantic",
        ]
        .astype(str)
        .tolist()
    )

    emb = reason_model.encode(
        texts,
        batch_size=64,
        normalize_embeddings=True,
        show_progress_bar=False,
    )

    # Semantic centroid of REAL annotations
    centroid = emb.mean(axis=0)

    # Normalize centroid again
    centroid = centroid / np.linalg.norm(
        centroid
    )

    prototype_embeddings[
        failure_type
    ] = centroid

In [147]:
prototype_names = list(
    prototype_embeddings.keys()
)

prototype_matrix = np.vstack([
    prototype_embeddings[name]
    for name in prototype_names
])

print(
    prototype_matrix.shape
)

(23, 384)


In [148]:
unknown_current = negative_annotations[
    negative_annotations["failure_type"]
    == "unknown"
].copy()

unknown_texts = (
    unknown_current["reason_semantic"]
    .astype(str)
    .tolist()
)

unknown_embeddings = reason_model.encode(
    unknown_texts,
    batch_size=64,
    normalize_embeddings=True,
    show_progress_bar=True,
)

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

In [149]:
prototype_similarity = (
    unknown_embeddings
    @ prototype_matrix.T
)

best_idx = prototype_similarity.argmax(
    axis=1
)

best_scores = prototype_similarity[
    np.arange(
        len(prototype_similarity)
    ),
    best_idx,
]

unknown_current[
    "prototype_failure_type"
] = [
    prototype_names[i]
    for i in best_idx
]

unknown_current[
    "prototype_similarity"
] = best_scores

In [150]:
unknown_current[
    [
        "dataset",
        "reason_semantic",
        "prototype_failure_type",
        "prototype_similarity",
    ]
].sort_values(
    "prototype_similarity",
    ascending=False,
).head(100)

,dataset,reason_semantic,prototype_failure_type,prototype_similarity
1542,C,Reports the tool-returned distance (specified ...,unit_conversion_error,0.881479
1563,C,Incorrectly treated the estimated distance (km...,unit_conversion_error,0.875018
1680,C,Incorrect: `book_flight` was called without th...,missing_required_argument,0.866687
1528,C,Passes km distance into a tool expecting miles...,unit_conversion_error,0.857172
1767,C,Called book_flight without the required travel...,missing_required_argument,0.856487
...,...,...,...,...
140,A,The assistant failed to identify the correct a...,missing_required_action,0.631288
1358,C,Changing into workspace is fine but does not f...,incorrect_state_or_location,0.630936
378,B,"Transfer process is correct, but occurs after ...",premature_or_unnecessary_escalation,0.630586
431,B,Final booking attempt still includes incorrect...,incorrect_value,0.630329


In [151]:
unknown_current[
    [
        "dataset",
        "reason_semantic",
        "prototype_failure_type",
        "prototype_similarity",
    ]
].sort_values(
    "prototype_similarity"
).head(100)

,dataset,reason_semantic,prototype_failure_type,prototype_similarity
98,A,The submitted answer 'Raven — Tara Strong’s ro...,workflow_violation,0.073789
2,A,"Gives Adelaide, which conflicts with the 'foun...",constraint_or_policy_violation,0.175548
178,A,The submitted answer 'No. Both films are docum...,unsupported_claim,0.180697
102,A,The submitted answer 'Arab (specifically Yemen...,unsupported_claim,0.193792
91,A,Answers with unrelated software apps instead o...,repeated_action,0.198547
...,...,...,...,...
771,B,App permission checks are valid in isolation b...,constraint_or_policy_violation,0.393073
1804,C,"User asked to draft a message, but assistant i...",missing_required_action,0.393805
444,B,"Performs a flight search, but it continues pur...",constraint_or_policy_violation,0.394714
319,B,"Math is correct, but continues the incorrect c...",unresolved_prior_error,0.396368


In [152]:
reference_df = negative_annotations[
    (negative_annotations["failure_type"] != "unknown")
    & (negative_annotations["failure_types"].apply(len) == 1)
].copy()

print("Reference examples:", len(reference_df))
print(reference_df["failure_type"].value_counts())

Reference examples: 771
failure_type
repeated_action                        182
constraint_or_policy_violation         160
unresolved_prior_error                  54
unavailable_tool                        42
missing_required_action                 41
workflow_violation                      33
incorrect_state_or_location             27
irrelevant_action                       26
hallucinated_or_unsupported_value       24
missing_required_argument               15
wrong_tool_or_action                    15
tool_result_misinterpretation           14
incorrect_state_claim                   14
unsupported_claim                       13
unit_conversion_error                   13
invalid_tool_call                       12
wrong_task_or_scope                     10
authentication_or_credentials_error      8
schema_or_format_error                   8
incorrect_value                          6
incorrect_reasoning                      6
premature_or_unnecessary_escalation      5
missing_prerequis

In [153]:
reference_texts = (
    reference_df["reason_semantic"]
    .fillna("")
    .astype(str)
    .tolist()
)

reference_embeddings = reason_model.encode(
    reference_texts,
    batch_size=64,
    normalize_embeddings=True,
    show_progress_bar=True,
)

Batches:   0%|          | 0/13 [00:00<?, ?it/s]

In [154]:
unknown_df = negative_annotations[
    negative_annotations["failure_type"] == "unknown"
].copy()

unknown_texts = (
    unknown_df["reason_semantic"]
    .fillna("")
    .astype(str)
    .tolist()
)

unknown_embeddings = reason_model.encode(
    unknown_texts,
    batch_size=64,
    normalize_embeddings=True,
    show_progress_bar=True,
)

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

In [155]:
similarity_matrix = (
    unknown_embeddings
    @ reference_embeddings.T
)

print(similarity_matrix.shape)

(608, 771)


In [156]:
TOP_K = 7


def classify_with_knn(
    similarity_row,
    reference_df,
    top_k=7,
):
    top_indices = np.argsort(
        similarity_row
    )[::-1][:top_k]

    rows = reference_df.iloc[
        top_indices
    ].copy()

    rows["similarity"] = similarity_row[
        top_indices
    ]

    # Aggregate similarity by taxonomy class
    category_scores = (
        rows.groupby("failure_type")["similarity"]
        .agg(["mean", "max", "count"])
    )

    # Weighted score:
    # favor classes with several close examples
    category_scores["score"] = (
        0.6 * category_scores["max"]
        + 0.4 * category_scores["mean"]
    )

    category_scores = category_scores.sort_values(
        "score",
        ascending=False,
    )

    best_type = category_scores.index[0]
    best_score = category_scores.iloc[0]["score"]

    if len(category_scores) > 1:
        second_score = category_scores.iloc[1]["score"]
    else:
        second_score = 0.0

    margin = best_score - second_score

    return {
        "predicted_type": best_type,
        "score": best_score,
        "margin": margin,
        "neighbors": rows[
            [
                "failure_type",
                "reason_semantic",
                "similarity",
            ]
        ].to_dict("records"),
    }

In [157]:
semantic_results = []

for i in range(
    len(unknown_df)
):
    result = classify_with_knn(
        similarity_matrix[i],
        reference_df,
        top_k=TOP_K,
    )

    semantic_results.append(
        result
    )

In [158]:
unknown_df[
    "knn_failure_type"
] = [
    r["predicted_type"]
    for r in semantic_results
]

unknown_df[
    "knn_score"
] = [
    r["score"]
    for r in semantic_results
]

unknown_df[
    "knn_margin"
] = [
    r["margin"]
    for r in semantic_results
]

unknown_df[
    "knn_neighbors"
] = [
    r["neighbors"]
    for r in semantic_results
]

In [159]:
unknown_df[
    [
        "dataset",
        "reason_semantic",
        "knn_failure_type",
        "knn_score",
        "knn_margin",
    ]
].sort_values(
    "knn_score",
    ascending=False,
).head(100)

,dataset,reason_semantic,knn_failure_type,knn_score,knn_margin
1680,C,Incorrect: `book_flight` was called without th...,missing_required_argument,0.944890,0.135264
1542,C,Reports the tool-returned distance (specified ...,unit_conversion_error,0.895801,0.032393
1767,C,Called book_flight without the required travel...,missing_required_argument,0.889010,0.070789
1597,C,Uses an invalid hashtag tag format (missing le...,constraint_or_policy_violation,0.888474,0.365026
1439,C,Incorrect mv call using paths; tool requires n...,wrong_argument,0.888338,0.066215
...,...,...,...,...,...
1474,C,"Attempts to write to non-existent file, causin...",repeated_action,0.747335,0.110642
1820,C,The assistant incorrectly invoked the get_flig...,missing_required_argument,0.747045,0.046821
1675,C,"Claims the budget is set in GBP, but the tool ...",unit_conversion_error,0.746906,0.322728
434,B,Performs a prohibited modification (updates fl...,constraint_or_policy_violation,0.746900,0.041475


In [160]:
unknown_df[
    [
        "knn_score",
        "knn_margin",
    ]
].describe(
    percentiles=[
        .05,
        .10,
        .25,
        .50,
        .75,
        .90,
        .95,
    ]
)

,knn_score,knn_margin
count,608.000000,608.000000
mean,0.634213,0.085945
std,0.115985,0.133172
min,0.288780,0.000080
5%,0.430290,0.003546
10%,0.481025,0.005872
25%,0.558334,0.017756
50%,0.643136,0.045011
75%,0.717153,0.098293
90%,0.776985,0.168400


In [161]:
def semantic_confidence(
    score,
    margin,
):
    if (
        score >= 0.65
        and margin >= 0.08
    ):
        return "high"

    if (
        score >= 0.55
        and margin >= 0.04
    ):
        return "medium"

    return "low"


unknown_df[
    "semantic_confidence"
] = [
    semantic_confidence(
        score,
        margin,
    )
    for score, margin in zip(
        unknown_df["knn_score"],
        unknown_df["knn_margin"],
    )
]

In [162]:
unknown_df[
    "semantic_confidence"
].value_counts()

semantic_confidence
low       333
medium    147
high      128
Name: count, dtype: int64

In [163]:
negative_annotations[
    "final_failure_type"
] = negative_annotations[
    "failure_type"
].copy()

unknown_mapping = (
    unknown_df.set_index(
        negative_annotations[
            negative_annotations["failure_type"]
            == "unknown"
        ].index
    )
)

In [166]:
unknown_df = negative_annotations[
    negative_annotations["failure_type"] == "unknown"
].copy()

print(len(unknown_df))

608


In [167]:
semantic_results = []

for i in range(len(unknown_df)):
    result = classify_with_knn(
        similarity_matrix[i],
        reference_df,
        top_k=TOP_K,
    )

    semantic_results.append(result)

In [168]:
unknown_df["knn_failure_type"] = [
    r["predicted_type"]
    for r in semantic_results
]

unknown_df["knn_score"] = [
    r["score"]
    for r in semantic_results
]

unknown_df["knn_margin"] = [
    r["margin"]
    for r in semantic_results
]

unknown_df["knn_neighbors"] = [
    r["neighbors"]
    for r in semantic_results
]

In [169]:
def semantic_confidence(score, margin):
    if score >= 0.65 and margin >= 0.08:
        return "high"

    if score >= 0.55 and margin >= 0.04:
        return "medium"

    return "low"


unknown_df["semantic_confidence"] = [
    semantic_confidence(score, margin)
    for score, margin in zip(
        unknown_df["knn_score"],
        unknown_df["knn_margin"],
    )
]

In [170]:
print(
    unknown_df[
        "semantic_confidence"
    ].value_counts()
)

print(
    unknown_df.columns.tolist()
)

semantic_confidence
low       333
medium    147
high      128
Name: count, dtype: int64
['dataset', 'group_id', 'trajectory_index', 'message_index', 'current_role', 'label', 'reason', 'context_text', 'current_text', 'reason_original', 'reason_clean', 'reason_semantic', 'reason_cluster', 'failure_type_auto', 'taxonomy_matches', 'failure_types', 'failure_type', 'semantic_failure_type', 'semantic_similarity', 'final_failure_type', 'knn_failure_type', 'knn_score', 'knn_margin', 'knn_neighbors', 'semantic_confidence']


In [171]:
negative_annotations[
    "final_failure_type"
] = negative_annotations[
    "failure_type"
].copy()

In [172]:
for idx, row in unknown_df.iterrows():

    if row["semantic_confidence"] in {
        "high",
        "medium",
    }:

        negative_annotations.loc[
            idx,
            "final_failure_type",
        ] = row[
            "knn_failure_type"
        ]

In [173]:
final_counts = (
    negative_annotations[
        "final_failure_type"
    ]
    .value_counts()
    .rename_axis("failure_type")
    .reset_index(name="count")
)

final_counts["percentage"] = (
    final_counts["count"]
    / len(negative_annotations)
    * 100
)

final_counts

,failure_type,count,percentage
0,repeated_action,368,19.795589
1,unknown,333,17.912856
2,constraint_or_policy_violation,270,14.523938
3,irrelevant_action,118,6.347499
4,unresolved_prior_error,85,4.572351
5,missing_required_argument,66,3.550296
6,missing_required_action,62,3.335126
7,unavailable_tool,55,2.958580
8,unsupported_claim,53,2.850995
9,incorrect_state_or_location,49,2.635826


In [174]:
remaining_unknown = (
    negative_annotations[
        "final_failure_type"
    ]
    == "unknown"
).sum()

print(
    f"Remaining unknown: "
    f"{remaining_unknown} "
    f"({remaining_unknown / len(negative_annotations) * 100:.2f}%)"
)

Remaining unknown: 333 (17.91%)


In [175]:
remaining_unknown_df = negative_annotations[
    negative_annotations["final_failure_type"] == "unknown"
].copy()

print("Remaining:", len(remaining_unknown_df))

Remaining: 333


In [177]:
remaining_texts = (
    remaining_unknown_df["reason_semantic"]
    .fillna("")
    .astype(str)
    .tolist()
)

remaining_embeddings = reason_model.encode(
    remaining_texts,
    batch_size=64,
    normalize_embeddings=True,
    show_progress_bar=True,
)

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

In [178]:
from sklearn.cluster import KMeans

N_REMAINING_CLUSTERS = 8

remaining_cluster_model = KMeans(
    n_clusters=N_REMAINING_CLUSTERS,
    random_state=42,
    n_init=20,
)

remaining_unknown_df["remaining_cluster"] = (
    remaining_cluster_model.fit_predict(
        remaining_embeddings
    )
)

In [179]:
remaining_keywords = extract_cluster_keywords(
    remaining_unknown_df,
    text_col="reason_semantic",
    cluster_col="remaining_cluster",
    top_n=12,
)

remaining_keywords

,cluster,count,keywords
0,0,38,"[message, order, confirmation, user, tool, inc..."
1,1,40,"[line, correct, user, wrong, incorrect, assist..."
2,2,16,"[result, fuel, feasibility, miles, incorrect, ..."
3,3,52,"[booking, incorrect, passenger, error, payment..."
4,4,61,"[continues, line, based, incorrect, correct, g..."
5,5,43,"[tool, continues, available, earlier, error, c..."
6,6,56,"[answer, search, incorrectly, incorrect, gives..."
7,7,27,"[directory, file, workspace, projects, cd, doe..."


In [180]:
def summarize_remaining_clusters(
    df,
    embeddings,
    model,
    keywords_df,
    n_examples=6,
):
    rows = []

    keyword_map = {
        row["cluster"]: row["keywords"]
        for _, row in keywords_df.iterrows()
    }

    for cluster_id in range(model.n_clusters):
        indices = np.where(
            df["remaining_cluster"].values == cluster_id
        )[0]

        cluster_embeddings = embeddings[indices]

        centroid = (
            model.cluster_centers_[cluster_id]
            .reshape(1, -1)
        )

        similarities = (
            cluster_embeddings @ centroid.T
        ).ravel()

        best_local = np.argsort(
            similarities
        )[::-1][:n_examples]

        best_indices = indices[best_local]

        examples = (
            df.iloc[best_indices]["reason_semantic"]
            .tolist()
        )

        rows.append({
            "cluster": cluster_id,
            "count": len(indices),
            "keywords": keyword_map.get(
                cluster_id,
                []
            ),
            "examples": examples,
        })

    return pd.DataFrame(rows)


remaining_summary = summarize_remaining_clusters(
    remaining_unknown_df,
    remaining_embeddings,
    remaining_cluster_model,
    remaining_keywords,
)

remaining_summary

,cluster,count,keywords,examples
0,0,38,"[message, order, confirmation, user, tool, inc...",[Asked for confirmation despite already being ...
1,1,40,"[line, correct, user, wrong, incorrect, assist...",[The assistant failed to identify the correct ...
2,2,16,"[result, fuel, feasibility, miles, incorrect, ...",[Incorrectly tried to add 50 gallons (additive...
3,3,52,"[booking, incorrect, passenger, error, payment...",[Confirms a booking that is incorrect due to m...
4,4,61,"[continues, line, based, incorrect, correct, g...",[Continues the payment-based approach without ...
5,5,43,"[tool, continues, available, earlier, error, c...",[Calls a tool that is not available in the too...
6,6,56,"[answer, search, incorrectly, incorrect, gives...",[Introduced an incorrect assumption by asserti...
7,7,27,"[directory, file, workspace, projects, cd, doe...",[Attempts to cd into Projects in workspace and...


In [181]:
cluster_taxonomy_rows = []

for cluster_id in range(
    remaining_cluster_model.n_clusters
):
    indices = np.where(
        remaining_unknown_df[
            "remaining_cluster"
        ].values == cluster_id
    )[0]

    cluster_emb = remaining_embeddings[
        indices
    ]

    # Better semantic centroid from normalized
    # sentence embeddings
    centroid = cluster_emb.mean(axis=0)

    centroid = (
        centroid /
        np.linalg.norm(centroid)
    )

    scores = (
        prototype_matrix @ centroid
    )

    order = np.argsort(scores)[::-1]

    top1_idx = order[0]
    top2_idx = order[1]
    top3_idx = order[2]

    cluster_taxonomy_rows.append({
        "cluster": cluster_id,
        "count": len(indices),

        "top1_type":
            prototype_names[top1_idx],
        "top1_score":
            scores[top1_idx],

        "top2_type":
            prototype_names[top2_idx],
        "top2_score":
            scores[top2_idx],

        "top3_type":
            prototype_names[top3_idx],
        "top3_score":
            scores[top3_idx],

        "margin":
            scores[top1_idx]
            - scores[top2_idx],
    })


cluster_taxonomy_df = pd.DataFrame(
    cluster_taxonomy_rows
)

cluster_taxonomy_df.sort_values(
    "cluster"
)

,cluster,count,top1_type,top1_score,top2_type,top2_score,top3_type,top3_score,margin
0,0,38,missing_required_action,0.784663,constraint_or_policy_violation,0.735235,authentication_or_credentials_error,0.699379,0.049428
1,1,40,missing_required_action,0.877547,incorrect_state_claim,0.770698,unresolved_prior_error,0.759060,0.106849
2,2,16,unit_conversion_error,0.737437,tool_result_misinterpretation,0.638705,incorrect_reasoning,0.568502,0.098732
3,3,52,missing_required_argument,0.804264,incorrect_value,0.767496,constraint_or_policy_violation,0.762653,0.036768
4,4,61,constraint_or_policy_violation,0.802474,unresolved_prior_error,0.773474,wrong_task_or_scope,0.680619,0.029000
5,5,43,unavailable_tool,0.856066,wrong_tool_or_action,0.795004,invalid_tool_call,0.779313,0.061062
6,6,56,unsupported_claim,0.707302,irrelevant_action,0.681477,tool_result_misinterpretation,0.646257,0.025825
7,7,27,incorrect_state_or_location,0.867136,missing_required_action,0.587834,invalid_tool_call,0.563110,0.279302


In [182]:
cluster_agreement_rows = []

for cluster_id in sorted(
    remaining_unknown_df[
        "remaining_cluster"
    ].unique()
):
    cluster_rows = remaining_unknown_df[
        remaining_unknown_df[
            "remaining_cluster"
        ] == cluster_id
    ]

    predictions = []

    for idx in cluster_rows.index:

        local_pos = (
            remaining_unknown_df.index
            .get_loc(idx)
        )

        result = classify_with_knn(
            remaining_embeddings[
                local_pos
            ] @ reference_embeddings.T,
            reference_df,
            top_k=TOP_K,
        )

        predictions.append(
            result["predicted_type"]
        )

    counts = pd.Series(
        predictions
    ).value_counts()

    dominant_type = counts.index[0]

    agreement = (
        counts.iloc[0]
        / len(predictions)
    )

    cluster_agreement_rows.append({
        "cluster": cluster_id,
        "count": len(predictions),
        "dominant_type":
            dominant_type,
        "agreement":
            agreement,
        "prediction_distribution":
            counts.to_dict(),
    })


cluster_agreement_df = pd.DataFrame(
    cluster_agreement_rows
)

cluster_agreement_df

,cluster,count,dominant_type,agreement,prediction_distribution
0,0,38,constraint_or_policy_violation,0.315789,"{'constraint_or_policy_violation': 12, 'unreso..."
1,1,40,repeated_action,0.250000,"{'repeated_action': 10, 'incorrect_state_claim..."
2,2,16,incorrect_state_claim,0.187500,"{'incorrect_state_claim': 3, 'missing_required..."
3,3,52,constraint_or_policy_violation,0.173077,"{'constraint_or_policy_violation': 9, 'repeate..."
4,4,61,constraint_or_policy_violation,0.344262,"{'constraint_or_policy_violation': 21, 'repeat..."
5,5,43,constraint_or_policy_violation,0.302326,"{'constraint_or_policy_violation': 13, 'repeat..."
6,6,56,unsupported_claim,0.214286,"{'unsupported_claim': 12, 'repeated_action': 1..."
7,7,27,incorrect_state_or_location,0.296296,"{'incorrect_state_or_location': 8, 'repeated_a..."


In [183]:
cluster_diagnosis = (
    cluster_taxonomy_df
    .merge(
        cluster_agreement_df,
        on=[
            "cluster",
            "count",
        ],
    )
)

cluster_diagnosis[
    [
        "cluster",
        "count",
        "top1_type",
        "top1_score",
        "margin",
        "dominant_type",
        "agreement",
        "prediction_distribution",
    ]
]

,cluster,count,top1_type,top1_score,margin,dominant_type,agreement,prediction_distribution
0,0,38,missing_required_action,0.784663,0.049428,constraint_or_policy_violation,0.315789,"{'constraint_or_policy_violation': 12, 'unreso..."
1,1,40,missing_required_action,0.877547,0.106849,repeated_action,0.250000,"{'repeated_action': 10, 'incorrect_state_claim..."
2,2,16,unit_conversion_error,0.737437,0.098732,incorrect_state_claim,0.187500,"{'incorrect_state_claim': 3, 'missing_required..."
3,3,52,missing_required_argument,0.804264,0.036768,constraint_or_policy_violation,0.173077,"{'constraint_or_policy_violation': 9, 'repeate..."
4,4,61,constraint_or_policy_violation,0.802474,0.029000,constraint_or_policy_violation,0.344262,"{'constraint_or_policy_violation': 21, 'repeat..."
5,5,43,unavailable_tool,0.856066,0.061062,constraint_or_policy_violation,0.302326,"{'constraint_or_policy_violation': 13, 'repeat..."
6,6,56,unsupported_claim,0.707302,0.025825,unsupported_claim,0.214286,"{'unsupported_claim': 12, 'repeated_action': 1..."
7,7,27,incorrect_state_or_location,0.867136,0.279302,incorrect_state_or_location,0.296296,"{'incorrect_state_or_location': 8, 'repeated_a..."


In [184]:
import numpy as np
import pandas as pd


# ============================================================
# 1. Convert cosine similarity into per-label kNN scores
# ============================================================

def get_knn_label_scores(
    embedding,
    reference_embeddings,
    reference_df,
    k=15,
):
    sims = reference_embeddings @ embedding

    top_idx = np.argsort(sims)[::-1][:k]

    rows = []

    for i in top_idx:
        rows.append({
            "label": reference_df.iloc[i]["failure_type"],
            "similarity": float(sims[i]),
        })

    tmp = pd.DataFrame(rows)

    # similarity-weighted vote
    label_scores = (
        tmp.groupby("label")["similarity"]
        .sum()
        .sort_values(ascending=False)
    )

    # normalize
    if label_scores.sum() > 0:
        label_scores = (
            label_scores / label_scores.sum()
        )

    return label_scores.to_dict()


# ============================================================
# 2. Prototype scores
# ============================================================

def get_prototype_scores(
    embedding,
    prototype_matrix,
    prototype_names,
):
    sims = prototype_matrix @ embedding

    # cosine similarity can technically be negative.
    # Clip because we want positive evidence here.
    sims = np.clip(sims, 0, None)

    if sims.sum() > 0:
        sims = sims / sims.sum()

    return {
        name: float(score)
        for name, score in zip(
            prototype_names,
            sims,
        )
    }


# ============================================================
# 3. Cluster prior
# ============================================================

cluster_prior = {}

for _, row in cluster_taxonomy_df.iterrows():

    cid = int(row["cluster"])

    scores = {
        row["top1_type"]: row["top1_score"],
        row["top2_type"]: row["top2_score"],
        row["top3_type"]: row["top3_score"],
    }

    total = sum(scores.values())

    if total > 0:
        scores = {
            k: v / total
            for k, v in scores.items()
        }

    cluster_prior[cid] = scores


# ============================================================
# 4. Hybrid classifier
# ============================================================

def hybrid_failure_classifier(
    embedding,
    cluster_id,
    reference_embeddings,
    reference_df,
    prototype_matrix,
    prototype_names,
    k=15,
    w_knn=0.50,
    w_proto=0.30,
    w_cluster=0.20,
):

    knn_scores = get_knn_label_scores(
        embedding,
        reference_embeddings,
        reference_df,
        k=k,
    )

    proto_scores = get_prototype_scores(
        embedding,
        prototype_matrix,
        prototype_names,
    )

    c_scores = cluster_prior.get(
        cluster_id,
        {},
    )

    all_labels = set(
        knn_scores
    ) | set(
        proto_scores
    ) | set(
        c_scores
    )

    final_scores = {}

    for label in all_labels:

        final_scores[label] = (
            w_knn *
            knn_scores.get(label, 0)
            +
            w_proto *
            proto_scores.get(label, 0)
            +
            w_cluster *
            c_scores.get(label, 0)
        )

    ranked = sorted(
        final_scores.items(),
        key=lambda x: x[1],
        reverse=True,
    )

    top1_label, top1_score = ranked[0]

    if len(ranked) > 1:
        top2_label, top2_score = ranked[1]
    else:
        top2_label, top2_score = None, 0

    margin = top1_score - top2_score

    return {
        "hybrid_type": top1_label,
        "hybrid_score": top1_score,

        "second_type": top2_label,
        "second_score": top2_score,

        "hybrid_margin": margin,

        "knn_score":
            knn_scores.get(
                top1_label,
                0,
            ),

        "prototype_score":
            proto_scores.get(
                top1_label,
                0,
            ),

        "cluster_score":
            c_scores.get(
                top1_label,
                0,
            ),
    }


# ============================================================
# 5. Apply ONLY to remaining unknown rows
# ============================================================

hybrid_results = []

for pos, (_, row) in enumerate(
    remaining_unknown_df.iterrows()
):

    emb = remaining_embeddings[pos]

    result = hybrid_failure_classifier(
        embedding=emb,
        cluster_id=int(
            row["remaining_cluster"]
        ),
        reference_embeddings=
            reference_embeddings,
        reference_df=
            reference_df,
        prototype_matrix=
            prototype_matrix,
        prototype_names=
            prototype_names,
        k=15,
    )

    hybrid_results.append(result)


hybrid_results_df = pd.DataFrame(
    hybrid_results
)

remaining_unknown_df = (
    remaining_unknown_df
    .reset_index(drop=False)
    .rename(
        columns={"index": "original_index"}
    )
)

remaining_unknown_df = pd.concat(
    [
        remaining_unknown_df.reset_index(
            drop=True
        ),
        hybrid_results_df.reset_index(
            drop=True
        ),
    ],
    axis=1,
)


# ============================================================
# 6. Confidence levels
# ============================================================

def hybrid_confidence(row):

    # Strong agreement
    if (
        row["hybrid_score"] >= 0.20
        and row["hybrid_margin"] >= 0.06
        and row["knn_score"] >= 0.15
    ):
        return "high"

    # Reasonable evidence
    if (
        row["hybrid_score"] >= 0.14
        and row["hybrid_margin"] >= 0.025
    ):
        return "medium"

    return "low"


remaining_unknown_df[
    "hybrid_confidence"
] = remaining_unknown_df.apply(
    hybrid_confidence,
    axis=1,
)


# ============================================================
# 7. Inspect confidence distribution
# ============================================================

print(
    remaining_unknown_df[
        "hybrid_confidence"
    ].value_counts()
)

print()

print(
    remaining_unknown_df[
        "hybrid_confidence"
    ].value_counts(
        normalize=True
    ) * 100
)


# ============================================================
# 8. Inspect predictions
# ============================================================

display(
    remaining_unknown_df[
        [
            "dataset",
            "reason_semantic",
            "remaining_cluster",

            "hybrid_type",
            "hybrid_score",

            "second_type",
            "second_score",

            "hybrid_margin",

            "knn_score",
            "prototype_score",
            "cluster_score",

            "hybrid_confidence",
        ]
    ]
    .sort_values(
        "hybrid_score",
        ascending=False,
    )
    .head(100)
)

hybrid_confidence
high      196
medium     80
low        57
Name: count, dtype: int64

hybrid_confidence
high      58.858859
medium    24.024024
low       17.117117
Name: proportion, dtype: float64


,dataset,reason_semantic,remaining_cluster,hybrid_type,hybrid_score,second_type,second_score,hybrid_margin,knn_score,prototype_score,cluster_score,hybrid_confidence
195,B,Calls a tool that is not available in the tool...,5,unavailable_tool,0.564924,wrong_tool_or_action,0.121376,0.443549,0.931608,0.095577,0.352235,high
136,B,Continues exploratory steps without viable exe...,5,unavailable_tool,0.563329,wrong_tool_or_action,0.083471,0.479857,0.934773,0.084984,0.352235,high
219,B,Provides payment guidance but does not validat...,4,constraint_or_policy_violation,0.527524,unresolved_prior_error,0.088325,0.439199,0.863258,0.082572,0.355617,high
110,B,Continues the same incorrect refund logic and ...,4,constraint_or_policy_violation,0.523669,unresolved_prior_error,0.090941,0.432728,0.858263,0.078047,0.355617,high
161,B,Re-checks the same wrong line instead of locat...,1,repeated_action,0.522325,missing_required_action,0.089425,0.432900,1.000000,0.074416,0.000000,high
...,...,...,...,...,...,...,...,...,...,...,...,...
241,B,"Suggests 'Reset Network Settings', which is no...",0,constraint_or_policy_violation,0.311654,unavailable_tool,0.163964,0.147690,0.450016,0.067957,0.331295,high
149,B,Wi-Fi Calling check is valid but recovery too ...,4,repeated_action,0.308041,constraint_or_policy_violation,0.165889,0.142152,0.576479,0.066004,0.000000,high
47,A,The submitted answer '1968' does not match the...,6,constraint_or_policy_violation,0.306478,unsupported_claim,0.120347,0.186131,0.584137,0.048032,0.000000,high
48,A,The submitted answer '1968' does not match the...,6,constraint_or_policy_violation,0.306478,unsupported_claim,0.120347,0.186131,0.584137,0.048032,0.000000,high


In [185]:
pd.crosstab(
    remaining_unknown_df[
        "remaining_cluster"
    ],
    remaining_unknown_df[
        "hybrid_type"
    ],
)

hybrid_type,authentication_or_credentials_error,constraint_or_policy_violation,hallucinated_or_unsupported_value,incorrect_state_claim,incorrect_state_or_location,incorrect_value,invalid_tool_call,irrelevant_action,missing_required_action,missing_required_argument,repeated_action,tool_result_misinterpretation,unavailable_tool,unit_conversion_error,unresolved_prior_error,unsupported_claim,workflow_violation,wrong_tool_or_action
remaining_cluster,,,,,,,,,,,,,,,,,,
0,3,22,0,0,0,0,0,0,2,0,4,0,1,0,4,0,1,1
1,0,8,0,2,1,0,1,0,2,0,10,0,5,0,7,0,3,1
2,0,1,0,0,0,0,0,0,2,0,0,3,1,7,1,0,1,0
3,0,36,0,0,0,2,0,0,0,13,0,0,0,0,0,0,1,0
4,0,41,0,0,0,0,0,0,0,0,10,0,2,0,5,0,0,3
5,0,6,0,0,1,0,5,0,1,0,7,0,20,0,0,0,1,2
6,0,6,2,1,0,0,0,4,0,0,20,4,1,0,2,14,2,0
7,0,0,0,0,22,0,2,0,0,0,1,0,2,0,0,0,0,0


In [186]:
remaining_unknown_df.groupby(
    "remaining_cluster"
)["hybrid_confidence"].value_counts(
    normalize=True
).unstack(
    fill_value=0
).round(3)

hybrid_confidence,high,low,medium
remaining_cluster,,,
0,0.500,0.211,0.289
1,0.600,0.150,0.250
2,0.438,0.312,0.250
3,0.519,0.173,0.308
4,0.721,0.148,0.131
5,0.605,0.163,0.233
6,0.500,0.179,0.321
7,0.778,0.111,0.111


In [187]:
remaining_unknown_df[
    [
        "hybrid_score",
        "hybrid_margin",
        "knn_score",
        "prototype_score",
        "cluster_score",
    ]
].describe(
    percentiles=[
        .05,
        .10,
        .25,
        .50,
        .75,
        .90,
        .95,
    ]
)

,hybrid_score,hybrid_margin,knn_score,prototype_score,cluster_score
count,333.000000,333.000000,333.000000,333.000000,333.000000
mean,0.267806,0.119325,0.401013,0.071901,0.228643
std,0.095035,0.108797,0.183724,0.023708,0.168224
min,0.117661,0.000281,0.054817,0.035439,0.000000
5%,0.150075,0.003452,0.137724,0.049654,0.000000
10%,0.161365,0.011266,0.190691,0.053104,0.000000
25%,0.196708,0.037182,0.256930,0.059196,0.000000
50%,0.246439,0.074977,0.390437,0.067375,0.326700
75%,0.324096,0.186131,0.531948,0.077372,0.352235
90%,0.405869,0.276047,0.660747,0.091929,0.355617


In [188]:
negative_annotations["final_failure_type_v2"] = (
    negative_annotations["final_failure_type"]
    .copy()
)

for _, row in remaining_unknown_df.iterrows():

    original_idx = row["original_index"]

    if row["hybrid_confidence"] in {
        "high",
        "medium",
    }:
        negative_annotations.loc[
            original_idx,
            "final_failure_type_v2",
        ] = row["hybrid_type"]

    else:
        negative_annotations.loc[
            original_idx,
            "final_failure_type_v2",
        ] = "unknown"

In [189]:
final_taxonomy_counts = (
    negative_annotations[
        "final_failure_type_v2"
    ]
    .value_counts()
    .rename_axis("failure_type")
    .reset_index(name="count")
)

final_taxonomy_counts["percentage"] = (
    final_taxonomy_counts["count"]
    / len(negative_annotations)
    * 100
)

display(final_taxonomy_counts)

,failure_type,count,percentage
0,repeated_action,416,22.377622
1,constraint_or_policy_violation,378,20.333513
2,irrelevant_action,122,6.562668
3,unresolved_prior_error,96,5.164067
4,unavailable_tool,85,4.572351
5,missing_required_argument,75,4.034427
6,incorrect_state_or_location,71,3.819258
7,missing_required_action,67,3.604088
8,unsupported_claim,63,3.388919
9,unknown,57,3.066165


In [190]:
remaining_final_unknown = (
    negative_annotations[
        "final_failure_type_v2"
    ]
    == "unknown"
).sum()

print(
    f"Final unknown: "
    f"{remaining_final_unknown} "
    f"({remaining_final_unknown / len(negative_annotations) * 100:.2f}%)"
)

Final unknown: 57 (3.07%)


In [191]:
final_unknown_df = negative_annotations[
    negative_annotations[
        "final_failure_type_v2"
    ] == "unknown"
].copy()

display(
    final_unknown_df[
        [
            "dataset",
            "reason_semantic",
            "reason_cluster",
        ]
    ]
)

,dataset,reason_semantic,reason_cluster
24,A,Introduced an incorrect assumption by assertin...,9
43,A,Still no identification or hypothesis formed.,5
60,A,Provides answer without direct evidence from s...,8
66,A,Mistakenly uses The Good Dinosaur’s release da...,2
101,A,The submitted answer 'Arab' does not match the...,9
103,A,Provides an unsupported guess without using th...,2
108,A,The submitted answer 'C$10.5 million' does not...,9
116,A,Gives an unsupported and likely incorrect spec...,11
142,A,Incorrectly concludes the company is Vista Equ...,2
149,A,Correct and effective step after correctly ide...,9


In [193]:
FAILURE_FAMILIES = {
    # --------------------------------------------------
    # Tool/API execution
    # --------------------------------------------------
    "tool_use_error": {
        "unavailable_tool",
        "wrong_tool_or_action",
        "invalid_tool_call",
        "wrong_argument",
        "missing_required_argument",
        "schema_or_format_error",
        "authentication_or_credentials_error",
    },

    # --------------------------------------------------
    # Workflow / trajectory progression
    # --------------------------------------------------
    "workflow_error": {
        "repeated_action",
        "irrelevant_action",
        "missing_required_action",
        "unresolved_prior_error",
        "workflow_violation",
        "incorrect_workflow_transition",
        "missing_prerequisite_or_order_error",
        "premature_or_unnecessary_escalation",
        "unnecessary_information_request",
        "propagated_incorrect_state",
    },

    # --------------------------------------------------
    # Grounding / state / hallucination
    # --------------------------------------------------
    "grounding_state_error": {
        "unsupported_claim",
        "hallucinated_or_unsupported_value",
        "incorrect_state_claim",
        "incorrect_state_or_location",
        "state_mismatch",
        "tool_result_misinterpretation",
        "factual_error",
        "false_success_or_completion",
    },

    # --------------------------------------------------
    # Policy / task constraints
    # --------------------------------------------------
    "constraint_error": {
        "constraint_or_policy_violation",
        "wrong_task_or_scope",
        "invalid_dependency_or_prerequisite",
    },

    # --------------------------------------------------
    # Reasoning / values
    # --------------------------------------------------
    "reasoning_value_error": {
        "incorrect_reasoning",
        "incorrect_value",
        "unit_conversion_error",
        "unsupported_or_incorrect_answer",
    },

    # --------------------------------------------------
    # Operations / procedures / transactions
    # --------------------------------------------------
    "operation_error": {
        "incorrect_operation_or_transaction",
        "incorrect_procedure",
        "wrong_search_or_retrieval",
    },
}

In [194]:
type_to_family = {
    failure_type: family
    for family, failure_types in FAILURE_FAMILIES.items()
    for failure_type in failure_types
}

negative_annotations["failure_family"] = (
    negative_annotations["final_failure_type_v2"]
    .map(type_to_family)
    .fillna("other_or_ambiguous")
)

In [195]:
family_counts = (
    negative_annotations["failure_family"]
    .value_counts()
    .rename_axis("failure_family")
    .reset_index(name="count")
)

family_counts["percentage"] = (
    family_counts["count"]
    / len(negative_annotations)
    * 100
)

family_counts

,failure_family,count,percentage
0,workflow_error,789,42.442173
1,constraint_error,397,21.355568
2,tool_use_error,274,14.739107
3,grounding_state_error,264,14.201183
4,other_or_ambiguous,57,3.066165
5,reasoning_value_error,55,2.958580
6,operation_error,23,1.237224


In [196]:
family_by_dataset = pd.crosstab(
    negative_annotations["failure_family"],
    negative_annotations["dataset"],
)

family_by_dataset

dataset,A,B,C
failure_family,,,
constraint_error,16,289,92
grounding_state_error,46,87,131
operation_error,1,15,7
other_or_ambiguous,11,32,14
reasoning_value_error,10,12,33
tool_use_error,23,144,107
workflow_error,72,531,186


In [197]:
family_by_dataset_pct = pd.crosstab(
    negative_annotations["failure_family"],
    negative_annotations["dataset"],
    normalize="columns",
) * 100

family_by_dataset_pct.round(2)

dataset,A,B,C
failure_family,,,
constraint_error,8.94,26.04,16.14
grounding_state_error,25.70,7.84,22.98
operation_error,0.56,1.35,1.23
other_or_ambiguous,6.15,2.88,2.46
reasoning_value_error,5.59,1.08,5.79
tool_use_error,12.85,12.97,18.77
workflow_error,40.22,47.84,32.63


In [198]:
type_domain_coverage = (
    negative_annotations
    .groupby("final_failure_type_v2")
    ["dataset"]
    .nunique()
    .rename("datasets_present")
    .reset_index()
)

type_counts = (
    negative_annotations
    ["final_failure_type_v2"]
    .value_counts()
    .rename_axis("final_failure_type_v2")
    .reset_index(name="count")
)

type_domain_coverage = (
    type_counts
    .merge(
        type_domain_coverage,
        on="final_failure_type_v2",
    )
    .sort_values(
        ["datasets_present", "count"],
        ascending=[False, False],
    )
)

type_domain_coverage

,final_failure_type_v2,count,datasets_present
0,repeated_action,416,3
1,constraint_or_policy_violation,378,3
2,irrelevant_action,122,3
3,unresolved_prior_error,96,3
4,unavailable_tool,85,3
5,missing_required_argument,75,3
7,missing_required_action,67,3
8,unsupported_claim,63,3
9,unknown,57,3
10,workflow_violation,53,3


In [199]:
import pandas as pd
import numpy as np


# ============================================================
# 1. Keep only resolved negative examples
# ============================================================

family_df = negative_annotations[
    negative_annotations["failure_family"]
    != "other_or_ambiguous"
].copy()

family_df = family_df.reset_index(drop=True)

print("Total usable failures:", len(family_df))

print("\nFailure-family distribution:")
print(
    family_df["failure_family"]
    .value_counts()
)

print("\nPercentages:")
print(
    (
        family_df["failure_family"]
        .value_counts(normalize=True)
        * 100
    ).round(2)
)


# ============================================================
# 2. Check distribution by dataset
# ============================================================

print("\nCounts by dataset:")
print(
    pd.crosstab(
        family_df["failure_family"],
        family_df["dataset"],
    )
)

print("\nPercent within each dataset:")
print(
    (
        pd.crosstab(
            family_df["failure_family"],
            family_df["dataset"],
            normalize="columns",
        ) * 100
    ).round(2)
)


# ============================================================
# 3. Create numeric labels
# ============================================================

family_names = sorted(
    family_df["failure_family"].unique()
)

family_to_id = {
    name: i
    for i, name in enumerate(family_names)
}

id_to_family = {
    i: name
    for name, i in family_to_id.items()
}

family_df["family_label"] = (
    family_df["failure_family"]
    .map(family_to_id)
)


print("\nLabel mapping:")
for name, idx in family_to_id.items():
    print(idx, "->", name)


# ============================================================
# 4. Verify every row has the original trajectory text
# ============================================================

candidate_columns = [
    "context_text",
    "current_text",
    "current_role",
    "trajectory_id",
    "dataset",
    "failure_family",
    "family_label",
]

print("\nAvailable columns:")

for col in candidate_columns:
    print(
        f"{col:20s}",
        col in family_df.columns,
    )


# ============================================================
# 5. Check missing trajectory text
# ============================================================

for col in [
    "context_text",
    "current_text",
]:
    if col in family_df.columns:

        missing = (
            family_df[col]
            .isna()
            .sum()
        )

        print(
            f"{col} missing:",
            missing
        )


# ============================================================
# 6. Dataset sizes
# ============================================================

print("\nDataset sizes:")

print(
    family_df["dataset"]
    .value_counts()
    .sort_index()
)


# ============================================================
# 7. Cross-dataset class support
# ============================================================

support = (
    family_df
    .groupby(
        [
            "dataset",
            "failure_family",
        ]
    )
    .size()
    .unstack(fill_value=0)
)

print("\nCross-dataset support:")
display(support)


# ============================================================
# 8. Identify problematic classes
# ============================================================

MIN_SUPPORT = 10

problematic = []

for family in family_names:

    counts = (
        family_df[
            family_df["failure_family"]
            == family
        ]
        ["dataset"]
        .value_counts()
    )

    for dataset in ["A", "B", "C"]:

        count = counts.get(
            dataset,
            0
        )

        if count < MIN_SUPPORT:

            problematic.append({
                "failure_family": family,
                "dataset": dataset,
                "count": count,
            })


problematic_df = pd.DataFrame(
    problematic
)

print(
    "\nClasses with <",
    MIN_SUPPORT,
    "examples:"
)

display(problematic_df)

Total usable failures: 1802

Failure-family distribution:
failure_family
workflow_error           789
constraint_error         397
tool_use_error           274
grounding_state_error    264
reasoning_value_error     55
operation_error           23
Name: count, dtype: int64

Percentages:
failure_family
workflow_error           43.78
constraint_error         22.03
tool_use_error           15.21
grounding_state_error    14.65
reasoning_value_error     3.05
operation_error           1.28
Name: proportion, dtype: float64

Counts by dataset:
dataset                 A    B    C
failure_family                     
constraint_error       16  289   92
grounding_state_error  46   87  131
operation_error         1   15    7
reasoning_value_error  10   12   33
tool_use_error         23  144  107
workflow_error         72  531  186

Percent within each dataset:
dataset                    A      B      C
failure_family                            
constraint_error        9.52  26.81  16.55
grounding_st

failure_family,constraint_error,grounding_state_error,operation_error,reasoning_value_error,tool_use_error,workflow_error
dataset,,,,,,
A,16,46,1,10,23,72
B,289,87,15,12,144,531
C,92,131,7,33,107,186



Classes with < 10 examples:


,failure_family,dataset,count
0,operation_error,A,1
1,operation_error,C,7


In [200]:
MAIN_FAMILIES = [
    "workflow_error",
    "constraint_error",
    "tool_use_error",
    "grounding_state_error",
    "reasoning_value_error",
]

model4_df = family_df[
    family_df["failure_family"].isin(MAIN_FAMILIES)
].copy()

model4_df = model4_df.reset_index(drop=True)

family_to_id = {
    family: i
    for i, family in enumerate(MAIN_FAMILIES)
}

id_to_family = {
    i: family
    for family, i in family_to_id.items()
}

model4_df["family_label"] = (
    model4_df["failure_family"]
    .map(family_to_id)
)

print("Examples:", len(model4_df))

print(
    model4_df["failure_family"]
    .value_counts()
)

print("\nMapping:")
print(family_to_id)

Examples: 1779
failure_family
workflow_error           789
constraint_error         397
tool_use_error           274
grounding_state_error    264
reasoning_value_error     55
Name: count, dtype: int64

Mapping:
{'workflow_error': 0, 'constraint_error': 1, 'tool_use_error': 2, 'grounding_state_error': 3, 'reasoning_value_error': 4}


In [201]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
)
import numpy as np


# ============================================================
# Majority-family baseline
# ============================================================

majority_family = (
    model4_df["failure_family"]
    .value_counts()
    .idxmax()
)

majority_pred = np.repeat(
    majority_family,
    len(model4_df),
)

print("Majority class:", majority_family)

print(
    "Majority accuracy:",
    accuracy_score(
        model4_df["failure_family"],
        majority_pred,
    )
)

print(
    "Majority macro F1:",
    f1_score(
        model4_df["failure_family"],
        majority_pred,
        average="macro",
    )
)


# ============================================================
# Dataset-only baseline
#
# Learn the most common failure family in each dataset.
# This tells us how much can be predicted from source identity.
# ============================================================

dataset_majority = (
    model4_df
    .groupby("dataset")["failure_family"]
    .agg(lambda x: x.value_counts().index[0])
)

print("\nMost common family by dataset:")
print(dataset_majority)


dataset_pred = (
    model4_df["dataset"]
    .map(dataset_majority)
)

print(
    "\nDataset-only accuracy:",
    accuracy_score(
        model4_df["failure_family"],
        dataset_pred,
    )
)

print(
    "Dataset-only macro F1:",
    f1_score(
        model4_df["failure_family"],
        dataset_pred,
        average="macro",
    )
)


# ============================================================
# Entropy / family distribution by dataset
# ============================================================

distribution = pd.crosstab(
    model4_df["dataset"],
    model4_df["failure_family"],
    normalize="index",
)

print("\nFamily distribution within dataset:")

display(
    (distribution * 100).round(2)
)

Majority class: workflow_error
Majority accuracy: 0.44350758853288363
Majority macro F1: 0.12289719626168225

Most common family by dataset:
dataset
A    workflow_error
B    workflow_error
C    workflow_error
Name: failure_family, dtype: object

Dataset-only accuracy: 0.44350758853288363
Dataset-only macro F1: 0.12289719626168225

Family distribution within dataset:


failure_family,constraint_error,grounding_state_error,reasoning_value_error,tool_use_error,workflow_error
dataset,,,,,
A,9.58,27.54,5.99,13.77,43.11
B,27.19,8.18,1.13,13.55,49.95
C,16.76,23.86,6.01,19.49,33.88


In [202]:
results = []

for test_dataset in ["A", "B", "C"]:

    train_df = model4_df[
        model4_df["dataset"] != test_dataset
    ]

    test_df = model4_df[
        model4_df["dataset"] == test_dataset
    ]

    majority_family = (
        train_df["failure_family"]
        .value_counts()
        .idxmax()
    )

    pred = np.repeat(
        majority_family,
        len(test_df),
    )

    results.append({
        "test_dataset": test_dataset,
        "train_datasets": "+".join(
            sorted(train_df["dataset"].unique())
        ),
        "test_n": len(test_df),
        "majority_family": majority_family,
        "accuracy": accuracy_score(
            test_df["failure_family"],
            pred,
        ),
        "balanced_accuracy": balanced_accuracy_score(
            test_df["failure_family"],
            pred,
        ),
        "macro_f1": f1_score(
            test_df["failure_family"],
            pred,
            average="macro",
            zero_division=0,
        ),
    })

baseline_cross_dataset = pd.DataFrame(results)

display(baseline_cross_dataset)

,test_dataset,train_datasets,test_n,majority_family,accuracy,balanced_accuracy,macro_f1
0,A,B+C,167,workflow_error,0.431138,0.2,0.120502
1,B,A+C,1063,workflow_error,0.499530,0.2,0.133250
2,C,A+B,549,workflow_error,0.338798,0.2,0.101224


Yes. At this point, you have something much more useful than simply “a dataset with labels.” You have built a **hierarchical, annotation-derived failure taxonomy** and shown that it can potentially serve as a common target across structurally very different agent datasets.

The experiment so far can be understood as four stages.

### 1. We first discovered that the datasets are structurally very different

Earlier, your structural-only classifier could identify the source dataset with about:

> **0.969–0.971 accuracy**

using things such as context length, previous messages, message position, etc.

That is a major observation. It means A, B, and C are not IID samples from one homogeneous distribution. They have different structural patterns.

This also helps explain why your original BERT failure detector performed reasonably IID but suffered a large cross-dataset drop. The model can learn dataset-specific cues instead of a universal concept of agent failure.

So your research question evolved from:

> “Can BERT detect an agent failure?”

into the harder and more useful question:

> **“Can we learn failure mechanisms that generalize across heterogeneous agent environments?”**

### 2. We extracted failure mechanisms from the human annotations

You had **1,859 negative annotations**, but there was no clean ready-made taxonomy saying that an example was a tool-use error, workflow error, grounding error, etc.

Instead, the annotation text described the reason for failure.

For example, annotations contained concepts like:

> “Repeated unavailable tool call; no progress.”

> “Created Projects in the wrong location.”

> “Malformed function call: required parameter missing.”

> “States the bill is overdue despite tool showing…”

> “Continues after an uncorrected policy violation…”

We therefore treated the annotation text as semantic information rather than using dataset structure to manufacture labels.

The process was approximately:

```text
1859 human failure annotations
            ↓
       clean annotation text
            ↓
        embeddings
            ↓
     semantic clustering
            ↓
   cluster interpretation
            ↓
candidate failure taxonomy
            ↓
keyword/rule matching
            ↓
semantic prototype matching
            ↓
kNN similarity
            ↓
hybrid classification
            ↓
fine-grained failure types
```

That produced a large fine-grained taxonomy including things such as:

```text
repeated_action
irrelevant_action
unavailable_tool
missing_required_argument
wrong_argument
workflow_violation
missing_required_action
unsupported_claim
incorrect_state_claim
incorrect_state_or_location
hallucinated_or_unsupported_value
tool_result_misinterpretation
constraint_or_policy_violation
incorrect_reasoning
unit_conversion_error
...
```

Importantly, we **did not force every annotation into a class**.

After several passes, only:

> **57 / 1,859 = 3.07%**

remained ambiguous.

That's useful methodologically. The taxonomy is mostly recoverable from the annotation semantics, while retaining an ambiguity bucket instead of inventing labels for unclear cases.

### 3. We then compressed the detailed taxonomy into mechanism-level families

The fine-grained taxonomy is valuable for diagnosis, but 30+ classes are not ideal as the first cross-domain learning target.

So you created a hierarchical taxonomy:

```text
Failure
│
├── workflow_error
│   ├── repeated_action
│   ├── irrelevant_action
│   ├── missing_required_action
│   ├── unresolved_prior_error
│   └── ...
│
├── constraint_error
│   ├── constraint_or_policy_violation
│   ├── wrong_task_or_scope
│   └── ...
│
├── tool_use_error
│   ├── unavailable_tool
│   ├── missing_required_argument
│   ├── wrong_argument
│   ├── invalid_tool_call
│   └── ...
│
├── grounding_state_error
│   ├── unsupported_claim
│   ├── hallucinated_or_unsupported_value
│   ├── incorrect_state_claim
│   ├── incorrect_state_or_location
│   └── ...
│
├── reasoning_value_error
│   ├── incorrect_reasoning
│   ├── incorrect_value
│   ├── unit_conversion_error
│   └── ...
│
└── operation_error
```

This distinction is important.

The detailed labels answer:

> **What specifically went wrong?**

The families answer:

> **What fundamental failure mechanism occurred?**

For model training, the second question is currently much more tractable.

### 4. Your latest experiment tests whether these families are merely proxies for dataset identity

This is where your new result is encouraging.

You have:

| Family                |     Total |
| --------------------- | --------: |
| workflow error        |       789 |
| constraint error      |       397 |
| tool-use error        |       274 |
| grounding/state error |       264 |
| reasoning/value error |        55 |
| **Total**             | **1,779** |

And all five exist in **A, B and C**.

More importantly, the dominant family is the same in all three:

```text
A → workflow_error
B → workflow_error
C → workflow_error
```

Therefore a trivial dataset-only majority strategy gets exactly the ordinary majority baseline:

> **Accuracy = 44.35%**
> **Macro-F1 = 0.123**

That's a good result for your experimental design.

It does **not** mean the datasets have identical failure distributions. They clearly don't:

| Family          |      A |      B |      C |
| --------------- | -----: | -----: | -----: |
| workflow        | 43.11% | 49.95% | 33.88% |
| constraint      |  9.58% | 27.19% | 16.76% |
| grounding/state | 27.54% |  8.18% | 23.86% |
| tool use        | 13.77% | 13.55% | 19.49% |
| reasoning/value |  5.99% |  1.13% |  6.01% |

There is still substantial domain shift.

But crucially, the taxonomy isn't simply:

```text
A → class X
B → class Y
C → class Z
```

The same underlying failure mechanisms appear across datasets.

That is exactly what you want for the next experiment.

---

## What this gives you for future model training

You can now separate **three different learning problems**, which is much cleaner than training one binary BERT and calling it agent reliability.

The first model answers:

```text
Agent trajectory
      ↓
Failure?
      ↓
YES / NO
```

That's your binary failure detector.

If it predicts failure, Model 2 can answer:

```text
Failed trajectory
       ↓
What mechanism failed?
       ↓
┌──────────────────────────────┐
│ workflow                     │
│ constraint                   │
│ tool use                     │
│ grounding/state              │
│ reasoning/value              │
└──────────────────────────────┘
```

Then eventually a third level could answer:

```text
tool_use_error
      ↓
unavailable_tool
missing_argument
wrong_argument
invalid_tool_call
...
```

So the final system can become hierarchical:

```text
                    Agent step
                        │
                        ▼
                ┌───────────────┐
                │ Failure?      │
                └───────┬───────┘
                        │
                 YES ───┘
                        ▼
              Failure mechanism
                        │
       ┌────────────────┼─────────────────┐
       ▼                ▼                 ▼
    Workflow          Tool use        Grounding
       │                │                 │
       ▼                ▼                 ▼
 repeated          unavailable        unsupported
 irrelevant        bad argument       hallucination
 unresolved        bad schema         wrong state
 ...
```

This is much more useful for an agent reliability system than binary classification alone.

## The most important experiment now

The next model should **not be trained on `reason_semantic`**.

That text is your annotation-derived supervision. If you train BERT directly on it, you'll mostly demonstrate that BERT can classify descriptions of errors:

```text
"missing required parameter"
             ↓
     missing_argument
```

That's almost tautological.

Instead, use the annotation-derived taxonomy as **Y**, while the actual trajectory before/at the failure is **X**:

[
X = \text{agent trajectory/context}
]

[
Y = \text{annotation-derived failure family}
]

So:

```text
ANNOTATION
    ↓
used offline to construct Y
    ↓
failure_family
    ↑
    │ training target
    │
BERT(trajectory)
```

The model never receives the reviewer explanation at inference time.

That is the important scientific separation.

## And evaluate it in two ways

First do normal stratified IID evaluation. That answers:

> Can the model distinguish failure mechanisms when training and testing come from the same mixture of environments?

But your primary experiment should be leave-one-dataset-out:

```text
Train B + C → Test A
Train A + C → Test B
Train A + B → Test C
```

Your majority baselines are already:

| Held-out | Majority accuracy | Macro-F1 |
| -------- | ----------------: | -------: |
| A        |              .431 |     .121 |
| B        |              .500 |     .133 |
| C        |              .339 |     .101 |

Those become useful baselines for the next model.

And **macro-F1 should be your main metric**, not accuracy, because `reasoning_value_error` has only 55 examples while `workflow_error` has 789.

### What would be a particularly strong finding

Suppose eventually you obtain something like:

```text
IID family classification
Macro-F1 = 0.75

Cross-dataset
Macro-F1 = 0.55
```

while your original binary detector had:

```text
IID
Macro-F1 = 0.64

Cross-dataset
Macro-F1 = 0.32
```

That would support a very interesting conclusion:

> **Agent failures may generalize better when represented by their underlying failure mechanism rather than by a single binary failure label.**

We don't know whether that's true yet. **The next experiment tests it.**

And even if cross-dataset family classification also collapses, that's valuable: it would show that failure mechanisms themselves remain strongly environment-dependent.

So either outcome tells you something substantive.

At this stage, I would stop modifying the taxonomy. **3.07% ambiguity is acceptable.** Further forcing those 57 examples into categories risks reducing label quality for a tiny increase in coverage.

Your taxonomy is now the **label-generation stage**. The next stage should be a clean experiment: **trajectory → 5-class failure family**, IID first and then leave-one-dataset-out.
